In [1]:
from typing import Any, Dict, List, Optional, Union, Callable, Type, Tuple, Set
import logging
from datetime import datetime

from langchain_core.messages import (
    AIMessage, 
    HumanMessage, 
    SystemMessage, 
    BaseMessage,
    ToolMessage
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import BaseTool, StructuredTool
from langgraph.graph import END
from langgraph.prebuilt.tool_node import ToolNode

from src.haive.agents.simple.agent import SimpleAgent, SimpleAgentConfig
from src.haive.core.engine.agent.agent import register_agent
from src.haive.core.engine.aug_llm import AugLLMConfig
from src.haive.core.models.llm.base import AzureLLMConfig

# Set up logging
logger = logging.getLogger(__name__)
from pydantic import BaseModel, Field

# =============================================
# Tool Configuration
# =============================================

class ToolConfig(BaseModel):
    """Configuration for a tool."""
    tool: Union[BaseTool, StructuredTool, Callable,BaseModel] = Field(
        ..., description="The tool implementation"
    )
    route_to: Optional[str] = Field(
        default=None, description="Where to route after this tool is used"
    )
    return_direct: bool = Field(
        default=False, 
        description="Whether to return directly to user"
    )
    timeout: Optional[float] = Field(
        default=None, description="Timeout in seconds"
    )
    retry_policy: Optional[Any] = Field(
        default=None, description="Retry policy for this tool"
    )
    fallback_response: Optional[str] = Field(
        default=None, description="Fallback response if tool fails"
    )


class NodeConfig(BaseModel):
    """Configuration for a custom node."""
    node_function: Callable = Field(..., description="The node function")
    node_name: str = Field(..., description="Name for this node")
    routes_to: Union[str, Dict[str, str], Any] = Field(
        ..., description="Where this node routes to"
    )


# =============================================
# React Agent Config
# =============================================

class ReactAgentConfig(SimpleAgentConfig):
    """
    Configuration for a React Agent.
    
    Extends SimpleAgentConfig with tools and routing capabilities.
    """
    # Node names
    reasoning_node_name: str = Field(
        default="agent", 
        description="Name for the reasoning node"
    )
    
    tool_node_name: str = Field(
        default="tools", 
        description="Name for the tool execution node"
    )
    
    structured_output_node_name: str = Field(
        default="structured_output", 
        description="Name for the structured output node"
    )
    
    # Tools configuration
    tools: List[Union[BaseTool, StructuredTool, Callable, ToolConfig]] = Field(
        default_factory=list, 
        description="List of tools available to the agent"
    )
    
    tool_choice: str = Field(
        default="auto", 
        description="Tool choice strategy: 'auto', 'any', or 'required'"
    )
    
    # Tool error handling
    handle_tool_errors: Union[bool, str, Callable] = Field(
        default=True,
        description="How to handle tool errors"
    )
    
    # Routing configuration
    router: Optional[Any] = Field(
        default=None,
        description="Optional custom router configuration"
    )
    
    # Custom nodes
    custom_nodes: List[NodeConfig] = Field(
        default_factory=list,
        description="Custom nodes to add to the graph"
    )
    
    # Structured output
    structured_output_model: Optional[Type[BaseModel]] = Field(
        default=None,
        description="Optional model for structured output"
    )
    
    structured_output_retry_policy: Optional[Any] = Field(
        default=None,
        description="Retry policy for structured output generation"
    )
    
    # Iteration control
    max_iterations: int = Field(
        default=10,
        description="Maximum number of iterations"
    )
    
    # State tracking
    track_iterations: bool = Field(
        default=True,
        description="Whether to track and limit iterations"
    )
    
    track_tool_calls: bool = Field(
        default=True,
        description="Whether to detect and track tool calls"
    )
    
    # Schema override for custom fields
    schema_extensions: Optional[Dict[str, Any]] = Field(
        default=None,
        description="Additional fields to add to the base schema"
    )
    
    @classmethod
    def from_aug_llm(cls,
                    aug_llm: AugLLMConfig,
                    tools: Optional[List[Union[BaseTool, StructuredTool, Callable, ToolConfig]]] = None,
                    structured_output_model: Optional[Type[BaseModel]] = None,
                    router: Optional[Any] = None,
                    custom_nodes: Optional[List[NodeConfig]] = None,
                    name: Optional[str] = None,
                    max_iterations: int = 10,
                    **kwargs) -> 'ReactAgentConfig':
        """
        Create a ReactAgentConfig from an existing AugLLMConfig.
        """
        return cls(
            name=name or f"react_agent_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
            engine=aug_llm,
            tools=tools or [],
            router=router,
            custom_nodes=custom_nodes or [],
            structured_output_model=structured_output_model,
            max_iterations=max_iterations,
            **kwargs
        )
    
    @classmethod
    def from_scratch(cls, 
                   system_prompt: Optional[str] = None,
                   model: str = "gpt-4o", 
                   temperature: float = 0.7,
                   tools: Optional[List[Union[BaseTool, StructuredTool, Callable, ToolConfig]]] = None,
                   structured_output_model: Optional[Type[BaseModel]] = None,
                   router: Optional[Any] = None,
                   custom_nodes: Optional[List[NodeConfig]] = None,
                   name: Optional[str] = None,
                   max_iterations: int = 10,
                   **kwargs) -> 'ReactAgentConfig':
        """
        Create a ReactAgentConfig from scratch.
        """
        # Use provided system prompt or default
        prompt_content = system_prompt or "You are a helpful assistant that can use tools."
        
        # Create prompt template
        messages = [
            SystemMessage(content=prompt_content),
            MessagesPlaceholder(variable_name="messages")
        ]
        prompt = ChatPromptTemplate.from_messages(messages)
        
        # Create LLM config
        llm_config = AzureLLMConfig(
            model=model,
            parameters={"temperature": temperature}
        )
        
        # Extract base tools from ToolConfig wrappers if needed
        processed_tools = []
        if tools:
            for tool in tools:
                if isinstance(tool, ToolConfig):
                    processed_tools.append(tool.tool)
                else:
                    processed_tools.append(tool)
        
        # Create AugLLM config
        aug_llm = AugLLMConfig(
            name=f"{name or 'react'}_llm",
            llm_config=llm_config,
            prompt_template=prompt,
            tools=processed_tools
        )
        
        # Create and return config
        return cls(
            name=name or f"react_agent_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
            engine=aug_llm,
            tools=tools or [],
            router=router,
            custom_nodes=custom_nodes or [],
            structured_output_model=structured_output_model,
            system_prompt=prompt_content,
            max_iterations=max_iterations,
            **kwargs
        )


# =============================================
# Helper function to prepare tools
# =============================================



# =============================================
# React Agent Implementation
# =============================================

@register_agent(ReactAgentConfig)
class ReactAgent(SimpleAgent):
    """
    React agent that implements the ReAct pattern for reasoning and tool use.
    """
    
    def __init__(self, config: ReactAgentConfig):
        """Initialize the ReactAgent with the provided configuration."""
        # Process tools and extract configurations before initializing the base agent
        self.processed_tools = []
        self.tool_configs = {}
        
        for item in config.tools:
            if isinstance(item, ToolConfig):
                # Extract the tool and keep the config
                tool = item.tool
                self.tool_configs[getattr(tool, "name", str(id(tool)))] = item
                self.processed_tools.append(tool)
                print(self.tool_configs)
            else:
                # Keep as is
                self.processed_tools.append(item.tool)
        
        # Fix tools for PydanticToolsParser
        self.processed_tools = prepare_tools_for_parser(self.processed_tools)
        
        # Update engine tools if engine is an AugLLMConfig
        if isinstance(config.engine, AugLLMConfig) and self.processed_tools:
            config.engine.tools = self.processed_tools
        
        # Create tool node if tools are provided
        if self.processed_tools:
            self.tool_node = ToolNode(
                self.processed_tools,
                name=config.tool_node_name,
                handle_tool_errors=config.handle_tool_errors
            )
        else:
            self.tool_node = None
            
        # Initialize base agent
        super().__init__(config)
    
    def setup_workflow(self) -> None:
        """Set up the ReAct workflow with reasoning, tools, and structured output."""
        # Extract configuration
        config = self.config
        has_tools = bool(config.tools)
        structured_output_model = config.structured_output_model
        
        # Create and add the reasoning node that handles iteration tracking internally
        self.graph.add_node(
            config.reasoning_node_name,
            self._create_reasoning_node()
        )
        
        # Set the entry point
        self.graph.set_entry_point(config.reasoning_node_name)
        
        # If tools are provided, set up tool execution
        if has_tools:
            # Add the tool node
            self.graph.add_node(config.tool_node_name, self.tool_node)
            
            # Determine the post-tool destination
            tool_destination = END if not structured_output_model else config.structured_output_node_name
            
            # If there's a custom router, use it
            if config.router:
                router_func = lambda state: config.router.get_route(state)
                
                # Create a mapping of all possible destinations
                destinations = {
                    route.destination: END if route.destination == "END" else route.destination
                    for route in config.router.routes
                }
                destinations[config.router.default_destination] = (
                    END if config.router.default_destination == "END" 
                    else config.router.default_destination
                )
                
                # Add the conditional edges
                self.graph.add_conditional_edges(
                    config.reasoning_node_name,
                    router_func,
                    destinations
                )
            else:
                # Use default router: tools or end
                self.graph.add_conditional_edges(
                    config.reasoning_node_name,
                    self._default_router,
                    {
                        config.tool_node_name: config.tool_node_name,
                        "end": tool_destination
                    }
                )
            
            # Set up tool routing based on tool configs
            tool_routes = {
                name: config.route_to 
                for name, config in self.tool_configs.items() 
                if config.route_to
            }
            
            if tool_routes:
                # Create a map of destinations to ensure they're valid
                destinations = {
                    route: END if route == "END" else route
                    for route in set(tool_routes.values())
                }
                
                # If reasoning_node isn't in destinations, add it as default
                if config.reasoning_node_name not in destinations:
                    destinations[config.reasoning_node_name] = config.reasoning_node_name
                
                # Add conditional edges from tools
                self.graph.add_conditional_edges(
                    config.tool_node_name,
                    self._create_tool_router(tool_routes),
                    destinations
                )
            else:
                # Default route from tools back to reasoning
                self.graph.add_edge(config.tool_node_name, config.reasoning_node_name)
        else:
            # Without tools, go straight to the end or structured output
            self.graph.add_edge(
                config.reasoning_node_name,
                END if not structured_output_model else config.structured_output_node_name
            )
        
        # If structured output is configured, add that node
        if structured_output_model:
            self.graph.add_node(
                config.structured_output_node_name,
                self._create_structured_output_node()
            )
            
            # Add edge from structured output to END
            self.graph.add_edge(config.structured_output_node_name, END)
        
        # Add any custom nodes
        for node_config in config.custom_nodes:
            # Add the node
            self.graph.add_node(node_config.node_name, node_config.node_function)
            
            # Set up routing
            if isinstance(node_config.routes_to, str):
                # Simple direct routing
                dest = node_config.routes_to
                self.graph.add_edge(node_config.node_name, END if dest == "END" else dest)
            elif isinstance(node_config.routes_to, dict):
                # Mapping-based routing
                def create_mapping_router(mapping):
                    def router(state):
                        # Simple key mapping
                        for key, dest in mapping.items():
                            if key in state:
                                return key
                        # Default to first key
                        return list(mapping.keys())[0]
                    return router
                
                # Convert string "END" to actual END
                routes = {
                    k: END if v == "END" else v 
                    for k, v in node_config.routes_to.items()
                }
                
                # Add conditional edges
                self.graph.add_conditional_edges(
                    node_config.node_name,
                    create_mapping_router(node_config.routes_to),
                    routes
                )
            else:
                # Router-based routing
                router_func = lambda state: node_config.routes_to.get_route(state)
                
                # Create a mapping of all possible destinations
                destinations = {
                    route.destination: END if route.destination == "END" else route.destination
                    for route in node_config.routes_to.routes
                }
                destinations[node_config.routes_to.default_destination] = (
                    END if node_config.routes_to.default_destination == "END" 
                    else node_config.routes_to.default_destination
                )
                
                # Add the conditional edges
                self.graph.add_conditional_edges(
                    node_config.node_name,
                    router_func,
                    destinations
                )
        
        logger.info(f"Set up ReAct workflow for {config.name}")
    
    def _create_reasoning_node(self) -> Callable:
        """Create the reasoning node that uses the engine and handles iteration tracking."""
        engine = self.engine
        config = self.config
        max_iterations = config.max_iterations
        track_iterations = config.track_iterations
        track_tool_calls = config.track_tool_calls
        
        def reasoning_node(state: Dict[str, Any]) -> Dict[str, Any]:
            """Reason about the current state and decide next steps."""
            # Track iterations if enabled
            if track_iterations:
                # Initialize remaining steps if not set
                if "remaining_steps" not in state:
                    state["remaining_steps"] = max_iterations
                    state["is_last_step"] = False
                else:
                    # Decrement remaining steps
                    state["remaining_steps"] -= 1
                    
                    # Check if we've reached the limit
                    if state["remaining_steps"] <= 0:
                        state["is_last_step"] = True
                        # Add a message about reaching max iterations if needed
                        messages = state.get("messages", [])
                        if messages:
                            last_message = messages[-1]
                            if not (isinstance(last_message, AIMessage) and "maximum number of iterations" in last_message.content):
                                state["messages"].append(
                                    AIMessage(content=f"I've reached the maximum number of iterations ({max_iterations}). "
                                             "I'll provide my best response based on what I've learned so far.")
                                )

            # Invoke the engine with the current state
            logger.debug(f"Invoking engine for reasoning in {config.name}")
            try:
                inputs = {**state}
                
                # For better error handling, we'll invoke the engine with try/except
                response = engine.invoke(inputs)
                
                # Check if the response contains a message directly
                if isinstance(response, AIMessage):
                    # Track tool calls if enabled
                    if track_tool_calls:
                        state["has_tool_calls"] = (
                            hasattr(response, "tool_calls") and bool(response.tool_calls)
                        )
                    
                    # Add to messages
                    state["messages"].append(response)
                    return state
                    
                # If response is a dict, merge with state
                elif isinstance(response, dict):
                    # Extract messages if present
                    if "messages" in response:
                        last_msg = None
                        if isinstance(response["messages"], list) and response["messages"]:
                            last_msg = response["messages"][-1]
                        
                        # Track tool calls if enabled and possible
                        if track_tool_calls and isinstance(last_msg, AIMessage):
                            state["has_tool_calls"] = (
                                hasattr(last_msg, "tool_calls") and bool(last_msg.tool_calls)
                            )
                    
                    # Merge response with state
                    state.update(response)
                    return state
                    
                # Handle other response types
                else:
                    # Convert to message and add to state
                    message = AIMessage(content=str(response))
                    state["messages"].append(message)
                    if track_tool_calls:
                        state["has_tool_calls"] = False
                    return state
            except Exception as e:
                # Log the error
                logger.error(f"Error in reasoning node: {str(e)}", exc_info=True)
                
                # Add error message to state
                error_msg = AIMessage(content=f"I encountered an error: {str(e)}")
                state["messages"].append(error_msg)
                state["has_tool_calls"] = False
                return state
        
        return reasoning_node
    
    def _default_router(self, state: Dict[str, Any]) -> str:
        """Default router for the reasoning node."""
        # If we're on the last step, go to the end
        if state.get("is_last_step", False) or state.get("remaining_steps", 0) <= 0:
            return "end"
        
        # Check if the last message has tool calls
        has_tool_calls = state.get("has_tool_calls", False)
        
        # If we have tool calls, route to the tool node
        if has_tool_calls:
            return self.config.tool_node_name
        
        # Otherwise, go to the end
        return "end"
    
    def _create_tool_router(self, tool_routes: Dict[str, str]) -> Callable:
        """Create a router for the tool node."""
        default_dest = self.config.reasoning_node_name
        
        def tool_router(state: Dict[str, Any]) -> str:
            """Route based on the name of the last executed tool."""
            messages = state.get("messages", [])
            
            # Find the most recent tool message
            for msg in reversed(messages):
                if isinstance(msg, ToolMessage) and hasattr(msg, "name"):
                    # If we have a route for this tool, use it
                    if msg.name in tool_routes:
                        return tool_routes[msg.name]
                    break
            
            # Default to reasoning node
            return default_dest
        
        return tool_router
    
    def _create_structured_output_node(self) -> Callable:
        """Create a node for generating structured output."""
        model = self.config.engine.llm_config.instantiate_llm()
        structured_output_model = self.config.structured_output_model
        retry_policy = self.config.structured_output_retry_policy
        
        def structured_output_node(state: Dict[str, Any]) -> Dict[str, Any]:
            """Generate structured output from the conversation."""
            messages = state.get("messages", [])
            
            # Get model with structured output
            model_with_structured_output = model.with_structured_output(structured_output_model)
            
            # Define the function to run with retry
            def generate_structured_output():
                return model_with_structured_output.invoke(messages)
            
            # Using the retry framework
            if retry_policy:
                from src.haive.core.graph.retry import execute_with_retry
                result = execute_with_retry(
                    generate_structured_output,
                    retry_policy=retry_policy,
                    fallback_result={}
                )
            else:
                # Simple execution without retry
                try:
                    result = generate_structured_output()
                except Exception as e:
                    logger.error(f"Error generating structured output: {str(e)}")
                    result = {}
                    # Add error message to state
                    state["messages"].append(
                        AIMessage(content=f"I had trouble generating a structured response: {str(e)}")
                    )
            
            # Add to state
            state["structured_response"] = result
            return state
        
        return structured_output_node


# =============================================
# Helper Functions
# =============================================

def configure_tool(
    tool: Union[BaseTool, StructuredTool, Callable],
    route_to: Optional[str] = None,
    return_direct: bool = False,
    timeout: Optional[float] = None,
    retry_policy: Optional[Any] = None,
    fallback_response: Optional[str] = None
) -> ToolConfig:
    """
    Configure a tool with routing and retry information.
    """
    return ToolConfig(
        tool=tool,
        route_to=route_to,
        return_direct=return_direct,
        timeout=timeout,
        retry_policy=retry_policy,
        fallback_response=fallback_response
    )


def create_node_config(
    node_function: Callable,
    node_name: str,
    route_to: Union[str, Dict[str, str], Any]
) -> NodeConfig:
    """
    Create a configuration for a custom node.
    """
    return NodeConfig(
        node_function=node_function,
        node_name=node_name,
        routes_to=route_to
    )


def create_react_agent(
    system_prompt: Optional[str] = None,
    model: str = "gpt-4o",
    temperature: float = 0.7,
    tools: Optional[List[Union[BaseTool, StructuredTool, Callable, ToolConfig]]] = None,
    structured_output_model: Optional[Type[BaseModel]] = None,
    router: Optional[Any] = None,
    custom_nodes: Optional[List[NodeConfig]] = None,
    name: Optional[str] = None,
    max_iterations: int = 10,
    retry_policy: Optional[Any] = None,
    visualize: bool = False,
    engine: Optional[AugLLMConfig] = None,
    **kwargs
) -> ReactAgent:
    """
    Create a React agent with the specified configuration.
    """
    if engine:
        # Create from existing AugLLMConfig
        config = ReactAgentConfig.from_aug_llm(
            aug_llm=engine,
            tools=tools,
            structured_output_model=structured_output_model,
            router=router,
            custom_nodes=custom_nodes,
            name=name,
            max_iterations=max_iterations,
            structured_output_retry_policy=retry_policy,
            should_visualize_graph=visualize,
            **kwargs
        )
    else:
        # Create from scratch
        config = ReactAgentConfig.from_scratch(
            system_prompt=system_prompt,
            model=model,
            temperature=temperature,
            tools=tools,
            structured_output_model=structured_output_model,
            router=router,
            custom_nodes=custom_nodes,
            name=name,
            max_iterations=max_iterations,
            structured_output_retry_policy=retry_policy,
            should_visualize_graph=visualize,
            **kwargs
        )
    
    # Build and return the agent
    return config.build_agent()

/home/will/Projects/haive/backend/haive/.venv/lib/python3.12/site-packages/pydantic/_internal/_config.py:345: UserWarning: Valid config keys have changed in V2:
* 'allow_population_by_field_name' has been renamed to 'populate_by_name'
* 'orm_mode' has been renamed to 'from_attributes'
  warnings.warn(message, UserWarning)


Setting system prompt to 'You are a helpful assistant.'
Setting system prompt to 'You are a helpful assistant.'


In [2]:
from typing import Any, Dict, List, Optional, Union, Callable, Type, Tuple, Set
import logging
import inspect
from datetime import datetime
import traceback
import json
import sys
import os

from langchain_core.messages import (
    AIMessage, 
    HumanMessage, 
    SystemMessage, 
    BaseMessage,
    ToolMessage
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import BaseTool, StructuredTool
from langgraph.graph import END
from langgraph.prebuilt.tool_node import ToolNode

from src.haive.agents.simple.agent import SimpleAgent, SimpleAgentConfig
from src.haive.core.engine.agent.agent import register_agent
from src.haive.core.engine.aug_llm import AugLLMConfig
from src.haive.core.models.llm.base import AzureLLMConfig

# Set up enhanced logging
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)
from pydantic import BaseModel, Field

# =============================================
# Helper functions for debugging
# =============================================

def debug_object(obj, prefix=""):
    """Print detailed information about an object for debugging."""
    obj_type = type(obj)
    obj_id = id(obj)
    
    logger.debug(f"{prefix}Object Type: {obj_type.__name__} (id: {obj_id})")
    
    if isinstance(obj, (str, int, float, bool, type(None))):
        logger.debug(f"{prefix}Value: {obj}")
    elif isinstance(obj, (list, tuple)):
        logger.debug(f"{prefix}Length: {len(obj)}")
        for i, item in enumerate(obj[:5]):  # Show first 5 items
            debug_object(item, prefix=f"{prefix}[{i}] ")
        if len(obj) > 5:
            logger.debug(f"{prefix}... and {len(obj) - 5} more items")
    elif isinstance(obj, dict):
        logger.debug(f"{prefix}Keys: {list(obj.keys())}")
        for k in list(obj.keys())[:5]:  # Show first 5 keys
            debug_object(obj[k], prefix=f"{prefix}['{k}'] ")
        if len(obj) > 5:
            logger.debug(f"{prefix}... and {len(obj) - 5} more keys")
    elif isinstance(obj, BaseModel):
        logger.debug(f"{prefix}Pydantic Model: {obj.__class__.__name__}")
        try:
            dict_repr = obj.dict()
            for k, v in list(dict_repr.items())[:5]:
                debug_object(v, prefix=f"{prefix}.{k} ")
            if len(dict_repr) > 5:
                logger.debug(f"{prefix}... and {len(dict_repr) - 5} more attributes")
        except:
            logger.debug(f"{prefix}Could not convert Pydantic model to dict")
    elif hasattr(obj, "__dict__"):
        attrs = vars(obj)
        logger.debug(f"{prefix}Attributes: {list(attrs.keys())}")
        for k in list(attrs.keys())[:5]:
            if k.startswith("_"):
                continue
            debug_object(attrs[k], prefix=f"{prefix}.{k} ")
        if len(attrs) > 5:
            logger.debug(f"{prefix}... and {len(attrs) - 5} more attributes")
    
    # Special handling for tools and callables
    if isinstance(obj, (BaseTool, StructuredTool)):
        logger.debug(f"{prefix}Tool details:")
        for attr in ["name", "description", "args_schema", "return_direct"]:
            if hasattr(obj, attr):
                logger.debug(f"{prefix}.{attr}: {getattr(obj, attr)}")
    elif callable(obj) and not isinstance(obj, type):
        logger.debug(f"{prefix}Callable: {obj.__name__ if hasattr(obj, '__name__') else 'Anonymous'}")
        sig = inspect.signature(obj)
        logger.debug(f"{prefix}Signature: {sig}")


def log_stack_trace():
    """Log the current stack trace for debugging."""
    stack = traceback.format_stack()
    logger.debug("Stack trace:")
    for line in stack[:-1]:  # Exclude the call to this function
        logger.debug(line.strip())

import logging
from typing import List, Any

logger = logging.getLogger(__name__)

def prepare_tools_for_parser(tools):
    """
    Prepares tools for the PydanticToolsParser by:
    1. Ensuring they have __name__ attributes
    2. Making sure they can handle direct argument passing correctly
    """
    for i, tool in enumerate(tools):
        logger.debug(f"Preparing tool {i+1}/{len(tools)}: {type(tool).__name__}")
        
        # Fix 1: Add __name__ attribute if missing
        if not hasattr(tool, "__name__") and hasattr(tool, "name"):
            logger.debug(f"  Adding __name__='{tool.name}' to tool (was missing)")
            setattr(tool, "__name__", tool.name)
        
        # Fix 2: Wrap the tool's __call__ method to handle the argument passing correctly
        if hasattr(tool, '__call__') and not getattr(tool, '_call_fixed', False):
            original_call = tool.__call__
            
            def fixed_call_method(self, *args, **kwargs):
                logger.debug(f"Fixed call for tool {self.name} with args={args}, kwargs={kwargs}")
                
                # If single keyword arg matches the schema's first parameter, extract it
                if hasattr(self, 'args_schema') and kwargs:
                    schema_params = getattr(self.args_schema, '__fields__', {})
                    if schema_params and len(kwargs) == 1:
                        param_name = next(iter(schema_params))
                        if param_name in kwargs and len(args) == 0:
                            # Convert to positional arg format that BaseTool expects
                            logger.debug(f"  Converting kwarg '{param_name}' to positional arg")
                            return original_call(kwargs[param_name])
                
                # Otherwise just call normally
                return original_call(*args, **kwargs)
            
            # Apply the fixed method
            tool.__call__ = fixed_call_method.__get__(tool)
            tool._call_fixed = True
            logger.debug(f"  Applied fixed __call__ method to tool {getattr(tool, 'name', str(id(tool)))}")
    
    return tools


def patch_structured_tool_run(tools):
    """
    Create fixed versions of StructuredTools by patching their run methods
    to handle arguments correctly.
    """
    from langchain_core.tools import StructuredTool
    
    for tool in tools:
        if isinstance(tool, StructuredTool) and not getattr(tool, '_run_fixed', False):
            original_run = tool.run
            
            def fixed_run(self, *args, **kwargs):
                logger.debug(f"Fixed run for StructuredTool {self.name} with args={args}, kwargs={kwargs}")
                
                # If we have keyword args that match our schema
                if kwargs and hasattr(self, 'args_schema'):
                    try:
                        # Try to validate the kwargs against the schema
                        validated_args = self.args_schema(**kwargs)
                        logger.debug(f"  Validated args with schema: {validated_args}")
                        # Call the function with the validated args as kwargs
                        return self.func(**validated_args.dict())
                    except Exception as e:
                        logger.error(f"  Error validating args: {e}")
                
                # Fall back to original behavior
                return original_run(*args, **kwargs)
            
            # Apply the fixed method
            tool.run = fixed_run.__get__(tool)
            tool._run_fixed = True
            logger.debug(f"  Applied fixed run method to StructuredTool {tool.name}")
    
    return tools


def fix_pydantic_tools_parser():
    """
    Monkey patch the PydanticToolsParser class to handle various tool formats correctly.
    This is a more invasive fix but might be necessary in some cases.
    """
    from langchain_core.output_parsers.openai_tools import PydanticToolsParser
    
    # Store original parse_result method
    original_parse_result = PydanticToolsParser.parse_result
    
    def fixed_parse_result(self, result, partial=False):
        logger.debug(f"Fixed PydanticToolsParser.parse_result called with result type: {type(result)}")
        
        try:
            # Create a safer name_dict that uses tool.name if __name__ is not available
            if not hasattr(self, '_fixed_name_dict'):
                name_dict = {}
                for tool in self.tools:
                    tool_name = getattr(tool, '__name__', None) or getattr(tool, 'name', None) or str(id(tool))
                    name_dict[tool_name] = tool
                self._fixed_name_dict = name_dict
                logger.debug(f"  Created fixed name_dict with keys: {list(name_dict.keys())}")
            
            # Process with original method but use our fixed name_dict
            # We'll use a temporary attribute swap to avoid modifying the class design
            original_dict = getattr(self, '_name_dict', None)
            setattr(self, '_name_dict', self._fixed_name_dict)
            
            try:
                return original_parse_result(self, result, partial)
            finally:
                # Restore original if it existed
                if original_dict is not None:
                    setattr(self, '_name_dict', original_dict)
                else:
                    delattr(self, '_name_dict')
                    
        except Exception as e:
            logger.error(f"Error in fixed PydanticToolsParser: {e}", exc_info=True)
            # Return empty list or None based on first_tool_only flag
            return None if self.first_tool_only else []
    
    # Apply the patch
    PydanticToolsParser.parse_result = fixed_parse_result
    logger.debug("Applied fix to PydanticToolsParser.parse_result")
    
    return True



# =============================================
# Tool Configuration
# =============================================

class ToolConfig(BaseModel):
    """Configuration for a tool."""
    tool: Union[BaseTool, StructuredTool, Callable, BaseModel] = Field(
        ..., description="The tool implementation"
    )
    route_to: Optional[str] = Field(
        default=None, description="Where to route after this tool is used"
    )
    return_direct: bool = Field(
        default=False, 
        description="Whether to return directly to user"
    )
    timeout: Optional[float] = Field(
        default=None, description="Timeout in seconds"
    )
    retry_policy: Optional[Any] = Field(
        default=None, description="Retry policy for this tool"
    )
    fallback_response: Optional[str] = Field(
        default=None, description="Fallback response if tool fails"
    )


class NodeConfig(BaseModel):
    """Configuration for a custom node."""
    node_function: Callable = Field(..., description="The node function")
    node_name: str = Field(..., description="Name for this node")
    routes_to: Union[str, Dict[str, str], Any] = Field(
        ..., description="Where this node routes to"
    )


# =============================================
# React Agent Config
# =============================================

class ReactAgentConfig(SimpleAgentConfig):
    """
    Configuration for a React Agent.
    
    Extends SimpleAgentConfig with tools and routing capabilities.
    """
    # Node names
    reasoning_node_name: str = Field(
        default="agent", 
        description="Name for the reasoning node"
    )
    
    tool_node_name: str = Field(
        default="tools", 
        description="Name for the tool execution node"
    )
    
    structured_output_node_name: str = Field(
        default="structured_output", 
        description="Name for the structured output node"
    )
    
    # Tools configuration
    tools: List[Union[BaseTool, StructuredTool, Callable, ToolConfig, BaseModel]] = Field(
        default_factory=list, 
        description="List of tools available to the agent"
    )
    
    tool_choice: str = Field(
        default="auto", 
        description="Tool choice strategy: 'auto', 'any', or 'required'"
    )
    
    # Tool error handling
    handle_tool_errors: Union[bool, str, Callable] = Field(
        default=True,
        description="How to handle tool errors"
    )
    
    # Routing configuration
    router: Optional[Any] = Field(
        default=None,
        description="Optional custom router configuration"
    )
    
    # Custom nodes
    custom_nodes: List[NodeConfig] = Field(
        default_factory=list,
        description="Custom nodes to add to the graph"
    )
    
    # Structured output
    structured_output_model: Optional[Type[BaseModel]] = Field(
        default=None,
        description="Optional model for structured output"
    )
    
    structured_output_retry_policy: Optional[Any] = Field(
        default=None,
        description="Retry policy for structured output generation"
    )
    
    # Iteration control
    max_iterations: int = Field(
        default=10,
        description="Maximum number of iterations"
    )
    
    # State tracking
    track_iterations: bool = Field(
        default=True,
        description="Whether to track and limit iterations"
    )
    
    track_tool_calls: bool = Field(
        default=True,
        description="Whether to detect and track tool calls"
    )
    
    # Schema override for custom fields
    schema_extensions: Optional[Dict[str, Any]] = Field(
        default=None,
        description="Additional fields to add to the base schema"
    )
    
    # Debugging level
    debug_level: int = Field(
        default=1,
        description="Debug level: 0=none, 1=basic, 2=detailed, 3=verbose"
    )
    
    @classmethod
    def from_aug_llm(cls,
                    aug_llm: AugLLMConfig,
                    tools: Optional[List[Union[BaseTool, StructuredTool, Callable, ToolConfig, BaseModel]]] = None,
                    structured_output_model: Optional[Type[BaseModel]] = None,
                    router: Optional[Any] = None,
                    custom_nodes: Optional[List[NodeConfig]] = None,
                    name: Optional[str] = None,
                    max_iterations: int = 10,
                    debug_level: int = 1,
                    **kwargs) -> 'ReactAgentConfig':
        """
        Create a ReactAgentConfig from an existing AugLLMConfig.
        """
        logger.debug(f"Creating ReactAgentConfig from AugLLMConfig, debug_level={debug_level}")
        if debug_level >= 2:
            debug_object(aug_llm, "aug_llm: ")
            if tools:
                for i, tool in enumerate(tools):
                    debug_object(tool, f"tools[{i}]: ")
        
        return cls(
            name=name or f"react_agent_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
            engine=aug_llm,
            tools=tools or [],
            router=router,
            custom_nodes=custom_nodes or [],
            structured_output_model=structured_output_model,
            max_iterations=max_iterations,
            debug_level=debug_level,
            **kwargs
        )
    
    @classmethod
    def from_scratch(cls, 
                   system_prompt: Optional[str] = None,
                   model: str = "gpt-4o", 
                   temperature: float = 0.7,
                   tools: Optional[List[Union[BaseTool, StructuredTool, Callable, ToolConfig, BaseModel]]] = None,
                   structured_output_model: Optional[Type[BaseModel]] = None,
                   router: Optional[Any] = None,
                   custom_nodes: Optional[List[NodeConfig]] = None,
                   name: Optional[str] = None,
                   max_iterations: int = 10,
                   debug_level: int = 1,
                   **kwargs) -> 'ReactAgentConfig':
        """
        Create a ReactAgentConfig from scratch.
        """
        logger.debug(f"Creating ReactAgentConfig from scratch, debug_level={debug_level}, model={model}")
        
        # Use provided system prompt or default
        prompt_content = system_prompt or "You are a helpful assistant that can use tools."
        logger.debug(f"Using system prompt: {prompt_content}")
        
        # Create prompt template
        messages = [
            SystemMessage(content=prompt_content),
            MessagesPlaceholder(variable_name="messages")
        ]
        prompt = ChatPromptTemplate.from_messages(messages)
        logger.debug(f"Created prompt template with {len(messages)} messages")
        
        # Create LLM config
        llm_config = AzureLLMConfig(
            model=model,
            parameters={"temperature": temperature}
        )
        logger.debug(f"Created LLM config with model={model}, temperature={temperature}")
        
        # Extract base tools from ToolConfig wrappers if needed
        processed_tools = []
        if tools:
            logger.debug(f"Processing {len(tools)} input tools")
            for i, tool in enumerate(tools):
                logger.debug(f"Tool {i+1}: type={type(tool).__name__}")
                
                if debug_level >= 2:
                    debug_object(tool, f"Input tool {i+1}: ")
                
                if isinstance(tool, ToolConfig):
                    logger.debug(f"  Tool {i+1} is a ToolConfig, extracting inner tool")
                    extracted_tool = tool.tool
                    processed_tools.append(extracted_tool)
                    
                    if debug_level >= 2:
                        debug_object(extracted_tool, f"  Extracted tool: ")
                else:
                    logger.debug(f"  Tool {i+1} is not a ToolConfig, using directly")
                    processed_tools.append(tool)
        
        # Create AugLLM config
        aug_llm = AugLLMConfig(
            name=f"{name or 'react'}_llm",
            llm_config=llm_config,
            prompt_template=prompt,
            tools=processed_tools
        )
        logger.debug(f"Created AugLLMConfig with {len(processed_tools)} tools")
        
        if debug_level >= 3:
            debug_object(aug_llm, "Final AugLLMConfig: ")
        
        # Create and return config
        return cls(
            name=name or f"react_agent_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
            engine=aug_llm,
            tools=tools or [],
            router=router,
            custom_nodes=custom_nodes or [],
            structured_output_model=structured_output_model,
            system_prompt=prompt_content,
            max_iterations=max_iterations,
            debug_level=debug_level,
            **kwargs
        )


# =============================================
# React Agent Implementation
# =============================================

@register_agent(ReactAgentConfig)
class ReactAgent(SimpleAgent):
    """
    React agent that implements the ReAct pattern for reasoning and tool use.
    """
    
    def __init__(self, config: ReactAgentConfig):
        """Initialize the ReactAgent with the provided configuration."""
        logger.debug(f"Initializing ReactAgent with config: {config.name}")
        self.debug_level = config.debug_level
        
        if self.debug_level >= 2:
            logger.debug("ReactAgent Config Details:")
            debug_object(config, "config: ")
        
        # Process tools and extract configurations before initializing the base agent
        self.processed_tools = []
        self.tool_configs = {}
        
        logger.debug(f"Processing {len(config.tools)} tools")
        
        for idx, item in enumerate(config.tools):
            logger.debug(f"Processing tool {idx+1}: {type(item).__name__}")
            
            if self.debug_level >= 2:
                debug_object(item, f"Tool {idx+1}: ")
            
            if isinstance(item, ToolConfig):
                # Extract the tool and keep the config
                logger.debug(f"Tool {idx+1} is a ToolConfig")
                tool = item.tool
                
                # Log tool details
                if hasattr(tool, "name"):
                    tool_name = tool.name
                    logger.debug(f"Tool {idx+1} name: {tool_name}")
                else:
                    tool_name = str(id(tool))
                    logger.debug(f"Tool {idx+1} has no name attribute, using id: {tool_name}")
                
                # Store the config
                self.tool_configs[getattr(tool, "name", str(id(tool)))] = item
                logger.debug(f"Stored tool config for {tool_name}")
                
                # Add the tool
                self.processed_tools.append(tool)
                logger.debug(f"Added tool {tool_name} to processed_tools")
                
                if self.debug_level >= 3:
                    logger.debug("Tool configs:")
                    for name, cfg in self.tool_configs.items():
                        logger.debug(f"  {name}: {cfg}")
            else:
                # Keep as is
                logger.debug(f"Tool {idx+1} is not a ToolConfig, using directly")
                
                # Try to log the tool name if available
                if hasattr(item, "name"):
                    logger.debug(f"Tool {idx+1} name: {item.name}")
                elif hasattr(item, "__name__"):
                    logger.debug(f"Tool {idx+1} __name__: {item.__name__}")
                
                self.processed_tools.append(item)
                logger.debug(f"Added tool to processed_tools")
        
        # Fix tools for PydanticToolsParser
        logger.debug(f"Preparing {len(self.processed_tools)} tools for PydanticToolsParser")
        self.processed_tools = prepare_tools_for_parser(self.processed_tools)
        
        # Update engine tools if engine is an AugLLMConfig
        if isinstance(config.engine, AugLLMConfig) and self.processed_tools:
            logger.debug(f"Updating engine tools with {len(self.processed_tools)} processed tools")
            
            if self.debug_level >= 2:
                logger.debug("Engine before tool update:")
                debug_object(config.engine, "engine: ")
            
            config.engine.tools = self.processed_tools
            
            if self.debug_level >= 2:
                logger.debug("Engine after tool update:")
                debug_object(config.engine, "engine: ")
        else:
            logger.debug(f"Engine is not AugLLMConfig or no tools to update: engine_type={type(config.engine).__name__}, tool_count={len(self.processed_tools)}")
        
        # Create tool node if tools are provided
        if self.processed_tools:
            logger.debug(f"Creating ToolNode with {len(self.processed_tools)} tools")
            self.tool_node = ToolNode(
                self.processed_tools,
                name=config.tool_node_name,
                handle_tool_errors=config.handle_tool_errors
            )
            logger.debug(f"Created ToolNode: {config.tool_node_name}")
        else:
            logger.debug("No tools provided, not creating ToolNode")
            self.tool_node = None
            
        # Initialize base agent
        logger.debug("Initializing base SimpleAgent")
        super().__init__(config)
        logger.debug("ReactAgent initialization complete")
    
    def setup_workflow(self) -> None:
        """Set up the ReAct workflow with reasoning, tools, and structured output."""
        logger.debug("Setting up ReactAgent workflow")
        
        # Extract configuration
        config = self.config
        has_tools = bool(config.tools)
        structured_output_model = config.structured_output_model
        
        logger.debug(f"Config: has_tools={has_tools}, has_structured_output={structured_output_model is not None}")
        
        # Create and add the reasoning node that handles iteration tracking internally
        logger.debug(f"Creating reasoning node: {config.reasoning_node_name}")
        reasoning_node = self._create_reasoning_node()
        self.graph.add_node(
            config.reasoning_node_name,
            reasoning_node
        )
        logger.debug(f"Added reasoning node: {config.reasoning_node_name}")
        
        # Set the entry point
        logger.debug(f"Setting entry point to reasoning node: {config.reasoning_node_name}")
        self.graph.set_entry_point(config.reasoning_node_name)
        
        # If tools are provided, set up tool execution
        if has_tools:
            logger.debug(f"Setting up tool execution with node: {config.tool_node_name}")
            
            # Add the tool node
            self.graph.add_node(config.tool_node_name, self.tool_node)
            logger.debug(f"Added tool node: {config.tool_node_name}")
            
            # Determine the post-tool destination
            tool_destination = END if not structured_output_model else config.structured_output_node_name
            logger.debug(f"Post-tool destination: {tool_destination}")
            
            # If there's a custom router, use it
            if config.router:
                logger.debug("Using custom router")
                router_func = lambda state: config.router.get_route(state)
                
                # Create a mapping of all possible destinations
                destinations = {
                    route.destination: END if route.destination == "END" else route.destination
                    for route in config.router.routes
                }
                destinations[config.router.default_destination] = (
                    END if config.router.default_destination == "END" 
                    else config.router.default_destination
                )
                
                logger.debug(f"Router destinations: {destinations}")
                
                # Add the conditional edges
                self.graph.add_conditional_edges(
                    config.reasoning_node_name,
                    router_func,
                    destinations
                )
                logger.debug(f"Added conditional edges from {config.reasoning_node_name} using custom router")
            else:
                # Use default router: tools or end
                logger.debug("Using default router (tools or end)")
                
                destinations = {
                    config.tool_node_name: config.tool_node_name,
                    "end": tool_destination
                }
                logger.debug(f"Default router destinations: {destinations}")
                
                self.graph.add_conditional_edges(
                    config.reasoning_node_name,
                    self._default_router,
                    destinations
                )
                logger.debug(f"Added conditional edges from {config.reasoning_node_name} using default router")
            
            # Set up tool routing based on tool configs
            tool_routes = {
                name: config.route_to 
                for name, config in self.tool_configs.items() 
                if config.route_to
            }
            
            logger.debug(f"Tool routes: {tool_routes}")
            
            if tool_routes:
                logger.debug("Setting up custom tool routing")
                # Create a map of destinations to ensure they're valid
                destinations = {
                    route: END if route == "END" else route
                    for route in set(tool_routes.values())
                }
                
                # If reasoning_node isn't in destinations, add it as default
                if config.reasoning_node_name not in destinations:
                    destinations[config.reasoning_node_name] = config.reasoning_node_name
                
                logger.debug(f"Tool routing destinations: {destinations}")
                
                # Add conditional edges from tools
                self.graph.add_conditional_edges(
                    config.tool_node_name,
                    self._create_tool_router(tool_routes),
                    destinations
                )
                logger.debug(f"Added conditional edges from {config.tool_node_name} using tool router")
            else:
                # Default route from tools back to reasoning
                logger.debug(f"Setting up default route from {config.tool_node_name} to {config.reasoning_node_name}")
                self.graph.add_edge(config.tool_node_name, config.reasoning_node_name)
        else:
            # Without tools, go straight to the end or structured output
            next_node = END if not structured_output_model else config.structured_output_node_name
            logger.debug(f"No tools, adding direct edge from {config.reasoning_node_name} to {next_node}")
            self.graph.add_edge(
                config.reasoning_node_name,
                next_node
            )
        
        # If structured output is configured, add that node
        if structured_output_model:
            logger.debug(f"Setting up structured output node: {config.structured_output_node_name}")
            self.graph.add_node(
                config.structured_output_node_name,
                self._create_structured_output_node()
            )
            logger.debug(f"Added structured output node: {config.structured_output_node_name}")
            
            # Add edge from structured output to END
            logger.debug(f"Adding edge from {config.structured_output_node_name} to END")
            self.graph.add_edge(config.structured_output_node_name, END)
        
        # Add any custom nodes
        for idx, node_config in enumerate(config.custom_nodes):
            logger.debug(f"Setting up custom node {idx+1}: {node_config.node_name}")
            
            # Add the node
            self.graph.add_node(node_config.node_name, node_config.node_function)
            logger.debug(f"Added custom node: {node_config.node_name}")
            
            # Set up routing
            if isinstance(node_config.routes_to, str):
                # Simple direct routing
                dest = node_config.routes_to
                dest_node = END if dest == "END" else dest
                logger.debug(f"Adding direct edge from {node_config.node_name} to {dest_node}")
                self.graph.add_edge(node_config.node_name, dest_node)
            elif isinstance(node_config.routes_to, dict):
                # Mapping-based routing
                logger.debug(f"Setting up mapping-based routing for {node_config.node_name}")
                
                def create_mapping_router(mapping):
                    def router(state):
                        # Simple key mapping
                        logger.debug(f"Mapping router state keys: {list(state.keys())}")
                        for key, dest in mapping.items():
                            if key in state:
                                logger.debug(f"Mapping router found key '{key}', routing to '{dest}'")
                                return key
                        # Default to first key
                        default_key = list(mapping.keys())[0]
                        logger.debug(f"Mapping router defaulting to first key: '{default_key}'")
                        return default_key
                    return router
                
                # Convert string "END" to actual END
                routes = {
                    k: END if v == "END" else v 
                    for k, v in node_config.routes_to.items()
                }
                
                logger.debug(f"Mapping routes: {routes}")
                
                # Add conditional edges
                self.graph.add_conditional_edges(
                    node_config.node_name,
                    create_mapping_router(node_config.routes_to),
                    routes
                )
                logger.debug(f"Added conditional edges from {node_config.node_name} using mapping router")
            else:
                # Router-based routing
                logger.debug(f"Setting up router-based routing for {node_config.node_name}")
                
                router_func = lambda state: node_config.routes_to.get_route(state)
                
                # Create a mapping of all possible destinations
                destinations = {
                    route.destination: END if route.destination == "END" else route.destination
                    for route in node_config.routes_to.routes
                }
                destinations[node_config.routes_to.default_destination] = (
                    END if node_config.routes_to.default_destination == "END" 
                    else node_config.routes_to.default_destination
                )
                
                logger.debug(f"Router destinations: {destinations}")
                
                # Add the conditional edges
                self.graph.add_conditional_edges(
                    node_config.node_name,
                    router_func,
                    destinations
                )
                logger.debug(f"Added conditional edges from {node_config.node_name} using router")
        
        logger.info(f"Set up ReAct workflow for {config.name}")
    
    def _create_reasoning_node(self) -> Callable:
        """Create the reasoning node that uses the engine and handles iteration tracking."""
        logger.debug("Creating reasoning node function")
        
        engine = self.engine
        config = self.config
        max_iterations = config.max_iterations
        track_iterations = config.track_iterations
        track_tool_calls = config.track_tool_calls
        debug_level = self.debug_level
        
        logger.debug(f"Reasoning node parameters: max_iterations={max_iterations}, track_iterations={track_iterations}, track_tool_calls={track_tool_calls}")
        
        def reasoning_node(state: Dict[str, Any]) -> Dict[str, Any]:
            """Reason about the current state and decide next steps."""
            node_id = f"reasoning_{id(state)}"
            logger.debug(f"[{node_id}] Reasoning node called with state keys: {list(state.keys())}")
            
            if debug_level >= 3:
                debug_object(state, f"[{node_id}] State: ")
            
            # Track iterations if enabled
            if track_iterations:
                logger.debug(f"[{node_id}] Tracking iterations")
                
                # Initialize remaining steps if not set
                if "remaining_steps" not in state:
                    logger.debug(f"[{node_id}] Initializing remaining_steps={max_iterations}")
                    state["remaining_steps"] = max_iterations
                    state["is_last_step"] = False
                else:
                    # Decrement remaining steps
                    state["remaining_steps"] -= 1
                    logger.debug(f"[{node_id}] Decremented remaining_steps to {state['remaining_steps']}")
                    
                    # Check if we've reached the limit
                    if state["remaining_steps"] <= 0:
                        logger.debug(f"[{node_id}] Reached max iterations, setting is_last_step=True")
                        state["is_last_step"] = True
                        # Add a message about reaching max iterations if needed
                        messages = state.get("messages", [])
                        if messages:
                            last_message = messages[-1]
                            if not (isinstance(last_message, AIMessage) and "maximum number of iterations" in last_message.content):
                                logger.debug(f"[{node_id}] Adding max iterations message")
                                state["messages"].append(
                                    AIMessage(content=f"I've reached the maximum number of iterations ({max_iterations}). "
                                             "I'll provide my best response based on what I've learned so far.")
                                )

            # Invoke the engine with the current state
            logger.debug(f"[{node_id}] Invoking engine for reasoning")
            try:
                # Copy state to preserve original in case of error
                inputs = {**state}
                logger.debug(f"[{node_id}] Engine input keys: {list(inputs.keys())}")
                
                if debug_level >= 3:
                    logger.debug(f"[{node_id}] Engine detail:")
                    debug_object(engine, f"[{node_id}] Engine: ")
                
                # For better error handling, we'll invoke the engine with try/except
                logger.debug(f"[{node_id}] Calling engine.invoke()")
                response = engine.invoke(inputs)
                logger.debug(f"[{node_id}] Engine response type: {type(response).__name__}")
                
                if debug_level >= 3:
                    debug_object(response, f"[{node_id}] Response: ")
                
                # Check if the response contains a message directly
                if isinstance(response, AIMessage):
                    logger.debug(f"[{node_id}] Response is AIMessage")
                    
                    # Track tool calls if enabled
                    if track_tool_calls:
                        has_calls = hasattr(response, "tool_calls") and bool(response.tool_calls)
                        state["has_tool_calls"] = has_calls
                        logger.debug(f"[{node_id}] has_tool_calls set to {has_calls}")
                        
                        if has_calls and debug_level >= 2:
                            logger.debug(f"[{node_id}] Tool calls in message: {response.tool_calls}")
                    
                    # Add to messages
                    state["messages"].append(response)
                    logger.debug(f"[{node_id}] Added AIMessage to state.messages")
                    return state
                    
                # If response is a dict, merge with state
                elif isinstance(response, dict):
                    logger.debug(f"[{node_id}] Response is dict with keys: {list(response.keys())}")
                    
                    # Extract messages if present
                    if "messages" in response:
                        last_msg = None
                        if isinstance(response["messages"], list) and response["messages"]:
                            last_msg = response["messages"][-1]
                            logger.debug(f"[{node_id}] Last message type: {type(last_msg).__name__}")
                        
                        # Track tool calls if enabled and possible
                        if track_tool_calls and isinstance(last_msg, AIMessage):
                            has_calls = hasattr(last_msg, "tool_calls") and bool(last_msg.tool_calls)
                            state["has_tool_calls"] = has_calls
                            logger.debug(f"[{node_id}] has_tool_calls set to {has_calls}")
                            
                            if has_calls and debug_level >= 2:
                                logger.debug(f"[{node_id}] Tool calls in message: {last_msg.tool_calls}")
                    
                    # Merge response with state
                    logger.debug(f"[{node_id}] Merging response dict with state")
                    state.update(response)
                    return state
                    
                # Handle other response types
                else:
                    logger.debug(f"[{node_id}] Response is other type: {type(response).__name__}, converting to AIMessage")
                    # Convert to message and add to state
                    message = AIMessage(content=str(response))
                    state["messages"].append(message)
                    if track_tool_calls:
                        state["has_tool_calls"] = False
                        logger.debug(f"[{node_id}] has_tool_calls set to False")
                    return state
                    
            except Exception as e:
                # Log the error
                logger.error(f"[{node_id}] Error in reasoning node: {str(e)}", exc_info=True)
                
                # Add error message to state
                error_msg = AIMessage(content=f"I encountered an error: {str(e)}")
                state["messages"].append(error_msg)
                state["has_tool_calls"] = False
                logger.debug(f"[{node_id}] Added error message to state and set has_tool_calls=False")
                return state
        
        logger.debug("Created reasoning node function")
        return reasoning_node
    
    def _default_router(self, state: Dict[str, Any]) -> str:
        """Default router for the reasoning node."""
        router_id = f"router_{id(state)}"
        logger.debug(f"[{router_id}] Default router called")
        
        # If we're on the last step, go to the end
        is_last = state.get("is_last_step", False) or state.get("remaining_steps", 0) <= 0
        if is_last:
            logger.debug(f"[{router_id}] Last step detected, routing to 'end'")
            return "end"
        
        # Check if the last message has tool calls
        has_tool_calls = state.get("has_tool_calls", False)
        logger.debug(f"[{router_id}] has_tool_calls = {has_tool_calls}")
        
        # If we have tool calls, route to the tool node
        if has_tool_calls:
            logger.debug(f"[{router_id}] Routing to tool node: {self.config.tool_node_name}")
            return self.config.tool_node_name
        
        # Otherwise, go to the end
        logger.debug(f"[{router_id}] No tool calls, routing to 'end'")
        return "end"
    
    def _create_tool_router(self, tool_routes: Dict[str, str]) -> Callable:
        """Create a router for the tool node."""
        logger.debug(f"Creating tool router with routes: {tool_routes}")
        default_dest = self.config.reasoning_node_name
        debug_level = self.debug_level
        
        def tool_router(state: Dict[str, Any]) -> str:
            """Route based on the name of the last executed tool."""
            router_id = f"tool_router_{id(state)}"
            messages = state.get("messages", [])
            logger.debug(f"[{router_id}] Tool router called, message count: {len(messages)}")
            
            if debug_level >= 3:
                logger.debug(f"[{router_id}] Last few messages:")
                for i, msg in enumerate(messages[-3:] if len(messages) >= 3 else messages):
                    logger.debug(f"[{router_id}] Message {len(messages) - 3 + i}: type={type(msg).__name__}")
            
            # Find the most recent tool message
            for i, msg in enumerate(reversed(messages)):
                if isinstance(msg, ToolMessage) and hasattr(msg, "name"):
                    logger.debug(f"[{router_id}] Found tool message with name: {msg.name}")
                    # If we have a route for this tool, use it
                    if msg.name in tool_routes:
                        destination = tool_routes[msg.name]
                        logger.debug(f"[{router_id}] Found route for '{msg.name}', routing to '{destination}'")
                        return destination
                    else:
                        logger.debug(f"[{router_id}] No specific route for '{msg.name}'")
                    break
                if i == 0:
                    logger.debug(f"[{router_id}] Last message type: {type(msg).__name__}, not a ToolMessage")
            
            # Default to reasoning node
            logger.debug(f"[{router_id}] Using default destination: {default_dest}")
            return default_dest
        
        logger.debug(f"Created tool router function")
        return tool_router
    
    def _create_structured_output_node(self) -> Callable:
        """Create a node for generating structured output."""
        logger.debug(f"Creating structured output node function")
        model = self.config.engine.llm_config.instantiate_llm()
        structured_output_model = self.config.structured_output_model
        retry_policy = self.config.structured_output_retry_policy
        debug_level = self.debug_level
        
        logger.debug(f"Structured output model: {structured_output_model.__name__ if structured_output_model else None}")
        logger.debug(f"Retry policy: {retry_policy}")
        
        def structured_output_node(state: Dict[str, Any]) -> Dict[str, Any]:
            """Generate structured output from the conversation."""
            node_id = f"structured_output_{id(state)}"
            logger.debug(f"[{node_id}] Structured output node called")
            
            messages = state.get("messages", [])
            logger.debug(f"[{node_id}] Processing {len(messages)} messages")
            
            if debug_level >= 3:
                for i, msg in enumerate(messages[-3:] if len(messages) >= 3 else messages):
                    logger.debug(f"[{node_id}] Message {len(messages) - 3 + i}: type={type(msg).__name__}")
            
            # Get model with structured output
            logger.debug(f"[{node_id}] Creating model with structured output")
            model_with_structured_output = model.with_structured_output(structured_output_model)
            
            # Define the function to run with retry
            def generate_structured_output():
                logger.debug(f"[{node_id}] Generating structured output")
                return model_with_structured_output.invoke(messages)
            
            # Using the retry framework
            if retry_policy:
                logger.debug(f"[{node_id}] Using retry policy")
                from src.haive.core.graph.retry import execute_with_retry
                result = execute_with_retry(
                    generate_structured_output,
                    retry_policy=retry_policy,
                    fallback_result={}
                )
                logger.debug(f"[{node_id}] Generated result with retry policy")
            else:
                # Simple execution without retry
                logger.debug(f"[{node_id}] Generating without retry policy")
                try:
                    result = generate_structured_output()
                    logger.debug(f"[{node_id}] Generated result successfully")
                except Exception as e:
                    logger.error(f"[{node_id}] Error generating structured output: {str(e)}", exc_info=True)
                    result = {}
                    # Add error message to state
                    state["messages"].append(
                        AIMessage(content=f"I had trouble generating a structured response: {str(e)}")
                    )
                    logger.debug(f"[{node_id}] Added error message to state")
            
            if debug_level >= 2:
                debug_object(result, f"[{node_id}] Structured output result: ")
            
            # Add to state
            state["structured_response"] = result
            logger.debug(f"[{node_id}] Added structured_response to state")
            return state
        
        logger.debug(f"Created structured output node function")
        return structured_output_node
    
    def run(self, input_text: str) -> Dict[str, Any]:
        """
        Override run to add extra debugging.
        """
        logger.debug(f"ReactAgent.run called with input: {input_text[:50]}...")
        
        if not self.app:
            logger.debug("Compiling workflow before run")
            self.compile_workflow(checkpointer=self.memory)
        
        inputs = {"messages": [("user", input_text)]}
        logger.debug(f"Created inputs: {inputs}")
        
        logger.debug("Invoking app with inputs")
        try:
            result = self.app.invoke(inputs, config=self.config.runtime_config, debug=True)
            logger.debug(f"Run completed successfully, result keys: {list(result.keys())}")
            
            if self.debug_level >= 3:
                debug_object(result, "Run result: ")
            
            return result
        except Exception as e:
            logger.error(f"Error in run: {str(e)}", exc_info=True)
            raise


# =============================================
# Helper Functions
# =============================================

def configure_tool(
    tool: Union[BaseTool, StructuredTool, Callable],
    route_to: Optional[str] = None,
    return_direct: bool = False,
    timeout: Optional[float] = None,
    retry_policy: Optional[Any] = None,
    fallback_response: Optional[str] = None
) -> ToolConfig:
    """
    Configure a tool with routing and retry information.
    """
    logger.debug(f"Configuring tool: {getattr(tool, 'name', str(id(tool)))}")
    logger.debug(f"  route_to: {route_to}")
    logger.debug(f"  return_direct: {return_direct}")
    logger.debug(f"  timeout: {timeout}")
    logger.debug(f"  has_retry_policy: {retry_policy is not None}")
    logger.debug(f"  fallback_response: {fallback_response[:30] + '...' if fallback_response and len(fallback_response) > 30 else fallback_response}")
    
    return ToolConfig(
        tool=tool,
        route_to=route_to,
        return_direct=return_direct,
        timeout=timeout,
        retry_policy=retry_policy,
        fallback_response=fallback_response
    )


def create_node_config(
    node_function: Callable,
    node_name: str,
    route_to: Union[str, Dict[str, str], Any]
) -> NodeConfig:
    """
    Create a configuration for a custom node.
    """
    logger.debug(f"Creating node config: {node_name}")
    
    if isinstance(route_to, str):
        logger.debug(f"  Direct routing to: {route_to}")
    elif isinstance(route_to, dict):
        logger.debug(f"  Mapping routing with keys: {list(route_to.keys())}")
    else:
        logger.debug(f"  Custom router of type: {type(route_to).__name__}")
    
    return NodeConfig(
        node_function=node_function,
        node_name=node_name,
        routes_to=route_to
    )


def create_react_agent(
    system_prompt: Optional[str] = None,
    model: str = "gpt-4o",
    temperature: float = 0.7,
    tools: Optional[List[Union[BaseTool, StructuredTool, Callable, ToolConfig, BaseModel]]] = None,
    structured_output_model: Optional[Type[BaseModel]] = None,
    router: Optional[Any] = None,
    custom_nodes: Optional[List[NodeConfig]] = None,
    name: Optional[str] = None,
    max_iterations: int = 10,
    retry_policy: Optional[Any] = None,
    visualize: bool = False,
    engine: Optional[AugLLMConfig] = None,
    debug_level: int = 1,
    **kwargs
) -> ReactAgent:
    """
    Create a React agent with the specified configuration.
    
    Args:
        system_prompt: Optional system prompt
        model: Model name to use
        temperature: Temperature for generation
        tools: Optional list of tools
        structured_output_model: Optional model for structured output
        router: Optional custom router
        custom_nodes: Optional custom nodes
        name: Optional agent name
        max_iterations: Maximum number of iterations
        retry_policy: Optional retry policy for structured output
        visualize: Whether to generate visualization
        engine: Optional existing AugLLMConfig to use
        debug_level: Debug level (0=none, 1=basic, 2=detailed, 3=verbose)
        **kwargs: Additional configuration options
        
    Returns:
        ReactAgent instance
    """
    logger.debug(f"Creating React agent: model={model}, debug_level={debug_level}")
    
    if engine:
        # Create from existing AugLLMConfig
        logger.debug("Creating from existing AugLLMConfig")
        config = ReactAgentConfig.from_aug_llm(
            aug_llm=engine,
            tools=tools,
            structured_output_model=structured_output_model,
            router=router,
            custom_nodes=custom_nodes,
            name=name,
            max_iterations=max_iterations,
            structured_output_retry_policy=retry_policy,
            should_visualize_graph=visualize,
            debug_level=debug_level,
            **kwargs
        )
    else:
        # Create from scratch
        logger.debug("Creating from scratch")
        config = ReactAgentConfig.from_scratch(
            system_prompt=system_prompt,
            model=model,
            temperature=temperature,
            tools=tools,
            structured_output_model=structured_output_model,
            router=router,
            custom_nodes=custom_nodes,
            name=name,
            max_iterations=max_iterations,
            structured_output_retry_policy=retry_policy,
            should_visualize_graph=visualize,
            debug_level=debug_level,
            **kwargs
        )
    
    # Build and return the agent
    logger.debug("Building agent from src.config")
    return config.build_agent()

In [3]:

def apply_all_fixes(agent_instance):
    """
    Apply all fixes to a ReactAgent instance.
    This should be called right after creating the agent.
    """
    logger.debug(f"Applying all fixes to ReactAgent: {agent_instance.__class__.__name__}")
    
    # Fix 1: Prepare tools for parser
    if hasattr(agent_instance, 'processed_tools'):
        agent_instance.processed_tools = prepare_tools_for_parser(agent_instance.processed_tools)
        logger.debug(f"Applied tool preparation fix to {len(agent_instance.processed_tools)} tools")
    
    # Fix 2: Patch StructuredTool.run method
    if hasattr(agent_instance, 'processed_tools'):
        agent_instance.processed_tools = patch_structured_tool_run(agent_instance.processed_tools)
        logger.debug("Applied StructuredTool.run patch")
    
    # Fix 3: Update the engine's tools reference
    if hasattr(agent_instance, 'config') and hasattr(agent_instance.config, 'engine'):
        config = agent_instance.config
        if hasattr(config.engine, 'tools') and hasattr(agent_instance, 'processed_tools'):
            config.engine.tools = agent_instance.processed_tools
            logger.debug(f"Updated engine tools reference with {len(agent_instance.processed_tools)} fixed tools")
    
    # Fix 4: Monkey patch the PydanticToolsParser
    fix_pydantic_tools_parser()
    
    logger.debug("All fixes applied successfully")
    return agent_instance


# Example code to integrate with create_react_agent
def create_fixed_react_agent(system_prompt=None, model="gpt-4o", tools=None, 
                            structured_output_model=None, **kwargs):
    """
    Create a React agent with all fixes applied.
    """
    from src.haive.agents.react_agent.agent import create_react_agent
    
    # Create the agent
    agent = create_react_agent(
        system_prompt=system_prompt,
        model=model,
        tools=tools,
        structured_output_model=structured_output_model,
        **kwargs
    )
    
    # Apply all fixes
    agent = apply_all_fixes(agent)
    
    return agent

In [4]:
import sys
import os
import logging
from datetime import datetime
from typing import Dict, Any, List

# Configure logging
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('react_agent_test.log')
    ]
)
logger = logging.getLogger(__name__)

# Add the src directory to the Python path if needed
# sys.path.append(os.path.abspath('src'))

from langchain_core.tools import tool, BaseTool, StructuredTool
from pydantic import BaseModel, Field

# Import the create_react_agent function


# =============================================
# Define test tools
# =============================================

@tool
def calculator(expression: str) -> str:
    """Calculate the result of a mathematical expression."""
    logger.info(f"Calculator tool called with: {expression}")
    try:
        result = eval(expression)
        return f"The result of {expression} is {result}"
    except Exception as e:
        logger.error(f"Calculator error: {str(e)}")
        return f"Error: {str(e)}"

@tool
def get_current_time() -> str:
    """Get the current time."""
    logger.info("Current time tool called")
    now = datetime.now()
    return f"The current time is {now.strftime('%H:%M:%S')}"

class WeatherInput(BaseModel):
    """Input for the weather tool."""
    location: str = Field(..., description="The city and state, e.g. San Francisco, CA")

@tool
def get_weather(location: str) -> str:
    """Get the weather forecast for a location."""
    logger.info(f"Weather tool called for: {location}")
    # This is a mock implementation
    return f"The weather in {location} is currently sunny with a temperature of 72°F."

# Create a customized tool with routing
calculator_with_routing = configure_tool(
    tool=calculator,
    route_to="agent",  # Go back to reasoning
    return_direct=False,
    timeout=5.0
)

# =============================================
# Define a structured output model
# =============================================

class Summary(BaseModel):
    """Summary of the conversation."""
    main_points: List[str] = Field(..., description="Main points from the conversation")
    conclusion: str = Field(..., description="Conclusion from the conversation")

# =============================================
# Test multiple scenarios
# =============================================

def test_simple_query():
    """Test a simple query with no tools."""
    logger.info("===== Testing Simple Query =====")
    
    agent = create_react_agent(
        system_prompt="You are a helpful assistant.",
        model="gpt-4o",
        name="simple_test",
        max_iterations=2,
        debug_level=2
    )
    
    result = agent.run("Tell me about the Python programming language in 2 sentences.")
    
    logger.info("===== Simple Query Result =====")
    for message in result.get("messages", []):
        if hasattr(message, "content"):
            logger.info(f"Message: {message.content}")

def test_with_tools():
    """Test using tools."""
    logger.info("===== Testing With Tools =====")
    
    agent = create_react_agent(
        system_prompt="You are a helpful assistant that can use tools.",
        model="gpt-4o",
        tools=[calculator, get_current_time, get_weather],
        name="tools_test",
        max_iterations=3,
        debug_level=2
    )
    
    result = agent.run("What is 42 * 18, and what time is it now? Also, what's the weather in Miami, FL?")
    
    logger.info("===== Tools Query Result =====")
    for message in result.get("messages", []):
        if hasattr(message, "content"):
            logger.info(f"Message: {message.content}")

def test_with_structured_output():
    """Test with structured output."""
    logger.info("===== Testing With Structured Output =====")
    
    agent = create_react_agent(
        system_prompt="You are a helpful assistant that can use tools.",
        model="gpt-4o",
        tools=[calculator, get_current_time],
        structured_output_model=Summary,
        name="structured_test",
        max_iterations=3,
        debug_level=2
    )
    
    result = agent.run("What is 144 divided by 12, and what time is it now?")
    
    logger.info("===== Structured Output Result =====")
    for message in result.get("messages", []):
        if hasattr(message, "content"):
            logger.info(f"Message: {message.content}")
    
    if "structured_response" in result:
        logger.info(f"Structured Response: {result['structured_response']}")

def test_with_configured_tools():
    """Test with configured tools that have routing info."""
    logger.info("===== Testing With Configured Tools =====")
    
    agent = create_react_agent(
        system_prompt="You are a helpful assistant that can use tools.",
        model="gpt-4o",
        tools=[calculator_with_routing, get_current_time],
        name="configured_tools_test",
        max_iterations=4,
        debug_level=3  # More verbose logging
    )
    
    result = agent.run("Calculate 25 * 4, then add 38 to that result.")
    
    logger.info("===== Configured Tools Result =====")
    for message in result.get("messages", []):
        if hasattr(message, "content"):
            logger.info(f"Message: {message.content}")

def main():
    """Run all tests."""
    logger.info("Starting ReactAgent debug tests")
    
    try:
        # Test a simple query first
        test_simple_query()
        
        # Test with tools
        test_with_tools()
        
        # Test with structured output
        test_with_structured_output()
        
        # Test with configured tools
        test_with_configured_tools()
        
        logger.info("All tests completed successfully")
    except Exception as e:
        logger.error(f"Test error: {str(e)}", exc_info=True)

if __name__ == "__main__":
    main()

2025-03-10 16:29:11,851 - __main__ - DEBUG - Configuring tool: calculator
2025-03-10 16:29:11,852 - __main__ - DEBUG -   route_to: agent
2025-03-10 16:29:11,852 - __main__ - DEBUG -   return_direct: False
2025-03-10 16:29:11,853 - __main__ - DEBUG -   timeout: 5.0
2025-03-10 16:29:11,854 - __main__ - DEBUG -   has_retry_policy: False
2025-03-10 16:29:11,854 - __main__ - DEBUG -   fallback_response: None
2025-03-10 16:29:11,856 - __main__ - INFO - Starting ReactAgent debug tests
2025-03-10 16:29:11,857 - __main__ - INFO - ===== Testing Simple Query =====
2025-03-10 16:29:11,857 - __main__ - DEBUG - Creating React agent: model=gpt-4o, debug_level=2
2025-03-10 16:29:11,858 - __main__ - DEBUG - Creating from scratch
2025-03-10 16:29:11,858 - __main__ - DEBUG - Creating ReactAgentConfig from scratch, debug_level=2, model=gpt-4o
2025-03-10 16:29:11,859 - __main__ - DEBUG - Using system prompt: You are a helpful assistant.
2025-03-10 16:29:11,859 - __main__ - DEBUG - Created prompt template w

/tmp/ipykernel_93491/2093883413.py:66: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  dict_repr = obj.dict()


2025-03-10 16:29:12,612 - urllib3.connectionpool - DEBUG - https://mermaid.ink:443 "GET /img/JSV7aW5pdDogeydmbG93Y2hhcnQnOiB7J2N1cnZlJzogJ2xpbmVhcid9fX0lJQpncmFwaCBURDsKCV9fc3RhcnRfXyhbPHA+X19zdGFydF9fPC9wPl0pOjo6Zmlyc3QKCWFnZW50KGFnZW50KQoJX19lbmRfXyhbPHA+X19lbmRfXzwvcD5dKTo6Omxhc3QKCV9fc3RhcnRfXyAtLT4gYWdlbnQ7CglhZ2VudCAtLT4gX19lbmRfXzsKCWNsYXNzRGVmIGRlZmF1bHQgZmlsbDojZjJmMGZmLGxpbmUtaGVpZ2h0OjEuMgoJY2xhc3NEZWYgZmlyc3QgZmlsbC1vcGFjaXR5OjAKCWNsYXNzRGVmIGxhc3QgZmlsbDojYmZiNmZjCg==?type=png&bgColor=!white HTTP/1.1" 200 5794
Graph diagram saved as /home/will/Projects/haive/backend/haive/resources/Graphs/simple_test_20250310_162911.png
2025-03-10 16:29:12,614 - src.haive.agents.base - INFO - Graph visualization saved to /home/will/Projects/haive/backend/haive/resources/Graphs/simple_test_20250310_162911.png
2025-03-10 16:29:12,615 - src.haive.agents.base - INFO - Agent simple_test initialized successfully
2025-03-10 16:29:12,616 - __main__ - DEBUG - ReactAgent initialization complete
2025

In [5]:
import logging
from typing import List, Any

logger = logging.getLogger(__name__)

def prepare_tools_for_parser(tools):
    """
    Prepares tools for the PydanticToolsParser by:
    1. Ensuring they have __name__ attributes
    2. Making sure they can handle direct argument passing correctly
    """
    for i, tool in enumerate(tools):
        logger.debug(f"Preparing tool {i+1}/{len(tools)}: {type(tool).__name__}")
        
        # Fix 1: Add __name__ attribute if missing
        if not hasattr(tool, "__name__") and hasattr(tool, "name"):
            logger.debug(f"  Adding __name__='{tool.name}' to tool (was missing)")
            setattr(tool, "__name__", tool.name)
        
        # Fix 2: Wrap the tool's __call__ method to handle the argument passing correctly
        if hasattr(tool, '__call__') and not getattr(tool, '_call_fixed', False):
            original_call = tool.__call__
            
            def fixed_call_method(self, *args, **kwargs):
                logger.debug(f"Fixed call for tool {self.name} with args={args}, kwargs={kwargs}")
                
                # If single keyword arg matches the schema's first parameter, extract it
                if hasattr(self, 'args_schema') and kwargs:
                    schema_params = getattr(self.args_schema, '__fields__', {})
                    if schema_params and len(kwargs) == 1:
                        param_name = next(iter(schema_params))
                        if param_name in kwargs and len(args) == 0:
                            # Convert to positional arg format that BaseTool expects
                            logger.debug(f"  Converting kwarg '{param_name}' to positional arg")
                            return original_call(kwargs[param_name])
                
                # Otherwise just call normally
                return original_call(*args, **kwargs)
            
            # Apply the fixed method
            tool.__call__ = fixed_call_method.__get__(tool)
            tool._call_fixed = True
            logger.debug(f"  Applied fixed __call__ method to tool {getattr(tool, 'name', str(id(tool)))}")
    
    return tools


def patch_structured_tool_run(tools):
    """
    Create fixed versions of StructuredTools by patching their run methods
    to handle arguments correctly.
    """
    from langchain_core.tools import StructuredTool
    
    for tool in tools:
        if isinstance(tool, StructuredTool) and not getattr(tool, '_run_fixed', False):
            original_run = tool.run
            
            def fixed_run(self, *args, **kwargs):
                logger.debug(f"Fixed run for StructuredTool {self.name} with args={args}, kwargs={kwargs}")
                
                # If we have keyword args that match our schema
                if kwargs and hasattr(self, 'args_schema'):
                    try:
                        # Try to validate the kwargs against the schema
                        validated_args = self.args_schema(**kwargs)
                        logger.debug(f"  Validated args with schema: {validated_args}")
                        # Call the function with the validated args as kwargs
                        return self.func(**validated_args.dict())
                    except Exception as e:
                        logger.error(f"  Error validating args: {e}")
                
                # Fall back to original behavior
                return original_run(*args, **kwargs)
            
            # Apply the fixed method
            tool.run = fixed_run.__get__(tool)
            tool._run_fixed = True
            logger.debug(f"  Applied fixed run method to StructuredTool {tool.name}")
    
    return tools


def fix_pydantic_tools_parser():
    """
    Monkey patch the PydanticToolsParser class to handle various tool formats correctly.
    This is a more invasive fix but might be necessary in some cases.
    """
    from langchain_core.output_parsers.openai_tools import PydanticToolsParser
    
    # Store original parse_result method
    original_parse_result = PydanticToolsParser.parse_result
    
    def fixed_parse_result(self, result, partial=False):
        logger.debug(f"Fixed PydanticToolsParser.parse_result called with result type: {type(result)}")
        
        try:
            # Create a safer name_dict that uses tool.name if __name__ is not available
            if not hasattr(self, '_fixed_name_dict'):
                name_dict = {}
                for tool in self.tools:
                    tool_name = getattr(tool, '__name__', None) or getattr(tool, 'name', None) or str(id(tool))
                    name_dict[tool_name] = tool
                self._fixed_name_dict = name_dict
                logger.debug(f"  Created fixed name_dict with keys: {list(name_dict.keys())}")
            
            # Process with original method but use our fixed name_dict
            # We'll use a temporary attribute swap to avoid modifying the class design
            original_dict = getattr(self, '_name_dict', None)
            setattr(self, '_name_dict', self._fixed_name_dict)
            
            try:
                return original_parse_result(self, result, partial)
            finally:
                # Restore original if it existed
                if original_dict is not None:
                    setattr(self, '_name_dict', original_dict)
                else:
                    delattr(self, '_name_dict')
                    
        except Exception as e:
            logger.error(f"Error in fixed PydanticToolsParser: {e}", exc_info=True)
            # Return empty list or None based on first_tool_only flag
            return None if self.first_tool_only else []
    
    # Apply the patch
    PydanticToolsParser.parse_result = fixed_parse_result
    logger.debug("Applied fix to PydanticToolsParser.parse_result")
    
    return True


def apply_all_fixes(agent_instance):
    """
    Apply all fixes to a ReactAgent instance.
    This should be called right after creating the agent.
    """
    logger.debug(f"Applying all fixes to ReactAgent: {agent_instance.__class__.__name__}")
    
    # Fix 1: Prepare tools for parser
    if hasattr(agent_instance, 'processed_tools'):
        agent_instance.processed_tools = prepare_tools_for_parser(agent_instance.processed_tools)
        logger.debug(f"Applied tool preparation fix to {len(agent_instance.processed_tools)} tools")
    
    # Fix 2: Patch StructuredTool.run method
    if hasattr(agent_instance, 'processed_tools'):
        agent_instance.processed_tools = patch_structured_tool_run(agent_instance.processed_tools)
        logger.debug("Applied StructuredTool.run patch")
    
    # Fix 3: Update the engine's tools reference
    if hasattr(agent_instance, 'config') and hasattr(agent_instance.config, 'engine'):
        config = agent_instance.config
        if hasattr(config.engine, 'tools') and hasattr(agent_instance, 'processed_tools'):
            config.engine.tools = agent_instance.processed_tools
            logger.debug(f"Updated engine tools reference with {len(agent_instance.processed_tools)} fixed tools")
    
    # Fix 4: Monkey patch the PydanticToolsParser
    fix_pydantic_tools_parser()
    
    logger.debug("All fixes applied successfully")
    return agent_instance


# Example code to integrate with create_react_agent
def create_fixed_react_agent(system_prompt=None, model="gpt-4o", tools=None, 
                            structured_output_model=None, **kwargs):
    """
    Create a React agent with all fixes applied.
    """
    from src.haive.agents.react_agent.agent import create_react_agent
    
    # Create the agent
    agent = create_react_agent(
        system_prompt=system_prompt,
        model=model,
        tools=tools,
        structured_output_model=structured_output_model,
        **kwargs
    )
    
    # Apply all fixes
    agent = apply_all_fixes(agent)
    
    return agent

In [6]:
# test_react_agent.py

import sys
import os

# Add the src directory to the Python path
#sys.path.append(os.path.abspath('src'))

from langchain_core.tools import tool
from pydantic import BaseModel, Field

# Import the create_react_agent function
#from src.haive.agents.react_agent.agent import create_react_agent

# Define some simple tools
@tool
def calculator(expression: str) -> str:
    """Calculate the result of a mathematical expression."""
    try:
        result = eval(expression)
        return f"The result of {expression} is {result}"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_current_time() -> str:
    """Get the current time."""
    from datetime import datetime
    now = datetime.now()
    return f"The current time is {now.strftime('%H:%M:%S')}"

# Create a simple structured output model
class Summary(BaseModel):
    main_points: list[str] = Field(..., description="Main points from the conversation")
    conclusion: str = Field(..., description="Conclusion from the conversation")

# Test the agent
def main():
    # Create the agent
    agent = create_react_agent(
        system_prompt="You are a helpful assistant that can use tools.",
        model="gpt-4o",
        tools=[calculator, get_current_time],
        structured_output_model=Summary,
        name="test_agent",
        max_iterations=5,
        visualize=True
    )
    
    # Run the agent
    print("\n===== Running agent =====\n")
    result = agent.run("What is 42 + 18, and what time is it now?")
    
    # Print the results
    print("\n===== Messages =====\n")
    for message in result.get("messages", []):
        if hasattr(message, "content"):
            print(f"AI: {message.content}")
        elif isinstance(message, tuple):
            print(f"{message[0].upper()}: {message[1]}")
        else:
            print(f"MESSAGE: {message}")
    
    if "structured_response" in result:
        print("\n===== Structured Response =====\n")
        print(result["structured_response"])

if __name__ == "__main__":
    main()

2025-03-10 16:29:12,656 - __main__ - DEBUG - Creating React agent: model=gpt-4o, debug_level=1
2025-03-10 16:29:12,657 - __main__ - DEBUG - Creating from scratch
2025-03-10 16:29:12,657 - __main__ - DEBUG - Creating ReactAgentConfig from scratch, debug_level=1, model=gpt-4o
2025-03-10 16:29:12,658 - __main__ - DEBUG - Using system prompt: You are a helpful assistant that can use tools.
2025-03-10 16:29:12,658 - __main__ - DEBUG - Created prompt template with 2 messages
2025-03-10 16:29:12,659 - __main__ - DEBUG - Created LLM config with model=gpt-4o, temperature=0.7
2025-03-10 16:29:12,660 - __main__ - DEBUG - Processing 2 input tools
2025-03-10 16:29:12,660 - __main__ - DEBUG - Tool 1: type=StructuredTool
2025-03-10 16:29:12,661 - __main__ - DEBUG -   Tool 1 is not a ToolConfig, using directly
2025-03-10 16:29:12,661 - __main__ - DEBUG - Tool 2: type=StructuredTool
2025-03-10 16:29:12,662 - __main__ - DEBUG -   Tool 2 is not a ToolConfig, using directly
2025-03-10 16:29:12,663 - __mai

AttributeError: 'SimpleAgentStateSchema' object has no attribute 'keys'

In [ ]:
# src/haive/agents/react_agent/agent.py (Part 1 - Config)

from __future__ import annotations
from typing import Any, Dict, List, Optional, Union, Callable, Type, Tuple, Set
from pydantic import BaseModel, Field, field_validator

from langchain_core.messages import (
    AIMessage, 
    SystemMessage,
    BaseMessage,
    ToolMessage,
    #MessagesPlaceholder
)
from langchain_core.prompts.chat import MessagesPlaceholder
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import BaseTool, StructuredTool
from langchain_core.runnables import Runnable

from langgraph.graph import StateGraph, END
from langgraph.prebuilt.tool_node import ToolNode

from src.haive.core.engine.agent.agent import Agent, AgentConfig, register_agent
from src.haive.core.models.llm.base import AzureLLMConfig
from src.haive.core.engine.aug_llm import AugLLMConfig
from src.haive.core.graph.NodeFactory import NodeFactory
from src.haive.core.graph.tool_injector import (
    ToolInjector, 
    state_tool, 
    store_tool, 
    hybrid_tool
)

# Import modules for React agent
from src.haive.core.graph.routing import (
    Router,
    RoutingCondition,
    Route,
    has_tool_calls,
    is_last_step
)
from src.haive.core.graph.retry import (
    RetryPolicy,
    execute_with_retry
)
from src.haive.core.graph.tool_config import (
    ToolConfig,
    NodeConfig,
    process_tools
)


# =============================================
# State Schemas
# =============================================

class ReactAgentSchema(BaseModel):
    """Base schema for React Agent state."""
    messages: List[Any] = Field(default_factory=list)
    remaining_steps: int = Field(default=10)
    is_last_step: bool = Field(default=False)
    has_tool_calls: bool = Field(default=False)


class ReactAgentStructuredSchema(ReactAgentSchema):
    """Schema for React Agent with structured output."""
    structured_response: Optional[Any] = Field(default=None)


# =============================================
# React Agent Config
# =============================================

class ReactAgentConfig(AgentConfig):
    """Configuration for a React Agent."""
    
    # Node names
    reasoning_node_name: str = Field(
        default="agent", 
        description="Name for the reasoning node"
    )
    
    tool_node_name: str = Field(
        default="tools", 
        description="Name for the tool execution node"
    )
    
    structured_output_node_name: str = Field(
        default="structured_output", 
        description="Name for the structured output node"
    )
    
    # Tools configuration
    tools: List[Union[BaseTool, StructuredTool, Callable, ToolConfig]] = Field(
        default_factory=list, 
        description="List of tools available to the agent"
    )
    
    tool_choice: str = Field(
        default="auto", 
        description="Tool choice strategy: 'auto', 'any', or 'required'"
    )
    
    # Tool error handling
    handle_tool_errors: Union[bool, str, Callable] = Field(
        default=True,
        description="How to handle tool errors"
    )
    
    # Custom nodes
    custom_nodes: List[NodeConfig] = Field(
        default_factory=list,
        description="Custom nodes to add to the graph"
    )
    
    # Custom routing
    custom_router: Optional[Router] = Field(
        default=None,
        description="Custom router configuration for complex routing"
    )
    
    # Structured output
    structured_output_model: Optional[Type[BaseModel]] = Field(
        default=None,
        description="Optional model for structured output"
    )
    
    structured_output_retry_policy: Optional[RetryPolicy] = Field(
        default=None,
        description="Retry policy for structured output generation"
    )
    
    # System prompt
    system_prompt: str = Field(
        default="""You are an intelligent agent that can use tools to help solve problems.
When you need information or want to perform an action, use the appropriate tool.
Think step-by-step and decide if you need to use any tools to fulfill the user's request.
If you need to use a tool, specify the tool name and parameters.
If you don't need a tool, simply respond directly to the user.""",
        description="System prompt for the agent"
    )
    
    # Iteration control
    max_iterations: int = Field(
        default=10,
        description="Maximum number of iterations"
    )
    
    # Override schema creation to detect structured output
    schema: Optional[Union[Type[BaseModel], Dict[str, Any]]] = Field(
        default=None,
        description="Optional explicit schema. If None, derived based on configuration."
    )
    
    @field_validator("schema", mode="before")
    def validate_schema(cls, v, info):
        """Auto-select the appropriate schema based on configuration."""
        if v is not None:
            return v
            
        # Use structured schema if structured output is configured
        if info.data.get("structured_output_model"):
            return ReactAgentStructuredSchema
        
        # Otherwise use the base schema
        return ReactAgentSchema

    @classmethod
    def from_scratch(cls, 
                   system_prompt: Optional[str] = None,
                   model: str = "gpt-4o", 
                   temperature: float = 0.7,
                   tools: Optional[List[Union[BaseTool, StructuredTool, Callable, ToolConfig]]] = None,
                   structured_output_model: Optional[Type[BaseModel]] = None,
                   custom_nodes: Optional[List[NodeConfig]] = None,
                   custom_router: Optional[Router] = None,
                   name: Optional[str] = None,
                   max_iterations: int = 10,
                   **kwargs) -> 'ReactAgentConfig':
        """
        Create a ReactAgentConfig from scratch with basic parameters.
        
        Args:
            system_prompt: Optional system prompt
            model: Model name to use
            temperature: Temperature for generation
            tools: Optional list of tools (can include ToolConfig for routing)
            structured_output_model: Optional model for structured output
            custom_nodes: Optional custom nodes to add to the graph
            custom_router: Optional custom router configuration
            name: Optional agent name
            max_iterations: Maximum number of iterations
            **kwargs: Additional configuration options
            
        Returns:
            ReactAgentConfig instance
        """
        # Use provided system prompt or default
        prompt_content = system_prompt or cls.__fields__["system_prompt"].default
        
        # Create prompt template
        messages = [
            SystemMessage(content=prompt_content),
            MessagesPlaceholder(variable_name="messages")
        ]
        prompt = ChatPromptTemplate.from_messages(messages)
        
        # Create LLM config
        llm_config = AzureLLMConfig(
            model=model,
            parameters={"temperature": temperature}
        )
        
        # Extract base tools from ToolConfig wrappers if needed
        processed_tools, _ = process_tools(tools or [])
        
        # Create AugLLM config
        aug_llm = AugLLMConfig(
            name=f"{name or 'react'}_llm",
            llm_config=llm_config,
            prompt_template=prompt,
            tools=processed_tools
        )
        
        # Create and return config
        return cls(
            name=name or "react_agent",
            engine=aug_llm,
            tools=tools or [],
            custom_nodes=custom_nodes or [],
            custom_router=custom_router,
            structured_output_model=structured_output_model,
            system_prompt=prompt_content,
            max_iterations=max_iterations,
            **kwargs
        )

    @classmethod
    def from_aug_llm(cls,
                     aug_llm: AugLLMConfig,
                     tools: Optional[List[Union[BaseTool, StructuredTool, Callable, ToolConfig]]] = None,
                     structured_output_model: Optional[Type[BaseModel]] = None,
                     custom_nodes: Optional[List[NodeConfig]] = None,
                     custom_router: Optional[Router] = None,
                     name: Optional[str] = None,
                     max_iterations: int = 10,
                     **kwargs) -> 'ReactAgentConfig':
        """
        Create a ReactAgentConfig using an existing AugLLMConfig.
        
        Args:
            aug_llm: Existing AugLLMConfig to use
            tools: Optional list of tools (can include ToolConfig for routing)
            structured_output_model: Optional model for structured output
            custom_nodes: Optional custom nodes to add to the graph
            custom_router: Optional custom router configuration
            name: Optional agent name
            max_iterations: Maximum number of iterations
            **kwargs: Additional configuration options
            
        Returns:
            ReactAgentConfig instance
        """
        # Update tools in AugLLMConfig if needed
        if tools:
            processed_tools, _ = process_tools(tools)
            
            # If the AugLLMConfig already has tools, we need to create a new one
            if aug_llm.tools:
                aug_llm = aug_llm.model_copy(update={"tools": processed_tools})
        
        # Create and return config
        return cls(
            name=name or "react_agent",
            engine=aug_llm,
            tools=tools or [],
            custom_nodes=custom_nodes or [],
            custom_router=custom_router,
            structured_output_model=structured_output_model,
            max_iterations=max_iterations,
            **kwargs
        )

    @classmethod
    def from_llm_config(cls,
                        llm_config: Union[AzureLLMConfig, Dict[str, Any]],
                        prompt_template: Optional[Union[str, ChatPromptTemplate]] = None,
                        tools: Optional[List[Union[BaseTool, StructuredTool, Callable, ToolConfig]]] = None,
                        structured_output_model: Optional[Type[BaseModel]] = None,
                        custom_nodes: Optional[List[NodeConfig]] = None,
                        custom_router: Optional[Router] = None,
                        name: Optional[str] = None,
                        max_iterations: int = 10,
                        **kwargs) -> 'ReactAgentConfig':
        """
        Create a ReactAgentConfig using an LLM configuration.
        
        Args:
            llm_config: LLM configuration (AzureLLMConfig or dict)
            prompt_template: Optional prompt template or system prompt string
            tools: Optional list of tools (can include ToolConfig for routing)
            structured_output_model: Optional model for structured output
            custom_nodes: Optional custom nodes to add to the graph
            custom_router: Optional custom router configuration
            name: Optional agent name
            max_iterations: Maximum number of iterations
            **kwargs: Additional configuration options
            
        Returns:
            ReactAgentConfig instance
        """
        # Handle LLM config
        if isinstance(llm_config, dict):
            llm_config = AzureLLMConfig(**llm_config)
        
        # Handle prompt template
        if prompt_template is None:
            # Use default system prompt
            system_prompt = cls.__fields__["system_prompt"].default
            prompt_template = ChatPromptTemplate.from_messages([
                SystemMessage(content=system_prompt),
                MessagesPlaceholder(variable_name="messages")
            ])
        elif isinstance(prompt_template, str):
            # Convert string to template
            prompt_template = ChatPromptTemplate.from_messages([
                SystemMessage(content=prompt_template),
                MessagesPlaceholder(variable_name="messages")
            ])
        
        # Process tools
        processed_tools, _ = process_tools(tools or [])
        
        # Create AugLLM config
        aug_llm = AugLLMConfig(
            name=f"{name or 'react'}_llm",
            llm_config=llm_config,
            prompt_template=prompt_template,
            tools=processed_tools
        )
        
        # Create and return config
        return cls(
            name=name or "react_agent",
            engine=aug_llm,
            tools=tools or [],
            custom_nodes=custom_nodes or [],
            custom_router=custom_router,
            structured_output_model=structured_output_model,
            max_iterations=max_iterations,
            **kwargs
        )

In [ ]:
# test_react_agent.py

import sys
import os

# Add the src directory to the Python path
#sys.path.append(os.path.abspath('src'))

from langchain_core.tools import tool
from pydantic import BaseModel, Field

# Import the create_react_agent function


# Define some simple tools
@tool
def calculator(expression: str) -> str:
    """Calculate the result of a mathematical expression."""
    try:
        result = eval(expression)
        return f"The result of {expression} is {result}"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_current_time() -> str:
    """Get the current time."""
    from datetime import datetime
    now = datetime.now()
    return f"The current time is {now.strftime('%H:%M:%S')}"

# Create a simple structured output model
class Summary(BaseModel):
    main_points: list[str] = Field(..., description="Main points from the conversation")
    conclusion: str = Field(..., description="Conclusion from the conversation")

# Test the agent
def main():
    # Create the agent
    agent = create_react_agent(
        system_prompt="You are a helpful assistant that can use tools.",
        model="gpt-4o",
        tools=[calculator, get_current_time],
        structured_output_model=Summary,
        name="test_agent",
        max_iterations=5,
        visualize=True
    )
    
    # Run the agent
    print("\n===== Running agent =====\n")
    #from src.haive.agents.react_agent.agent import create_react_agent
    #from tool_fix_complete import apply_all_fixes
    
    # Create your agent normally
    agent = create_react_agent(
        system_prompt="You are a helpful assistant that can use tools.",
        model="gpt-4o",
        tools=[calculator, get_current_time],
        structured_output_model=Summary,
        name="test_agent",
        max_iterations=5
    )
    
    # Apply all fixes
    agent = apply_all_fixes(agent)
    
    # Now the agent should work correctly
    result = agent.run("What is 42 + 18, and what time is it now?")
    
    # Print the results
    print("\n===== Messages =====\n")
    for message in result.get("messages", []):
        if hasattr(message, "content"):
            print(f"AI: {message.content}")
        elif isinstance(message, tuple):
            print(f"{message[0].upper()}: {message[1]}")
        else:
            print(f"MESSAGE: {message}")
    
    if "structured_response" in result:
        print("\n===== Structured Response =====\n")
        print(result["structured_response"])

if __name__ == "__main__":
    main()

In [ ]:
# src/haive/agents/react_agent/agent.py (Part 1 - Config)

from __future__ import annotations
from typing import Any, Dict, List, Optional, Union, Callable, Type, Tuple, Set
from pydantic import BaseModel, Field, field_validator

from langchain_core.messages import (
    AIMessage, 
    SystemMessage,
    BaseMessage,
    ToolMessage,
    
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import BaseTool, StructuredTool
from langchain_core.runnables import Runnable

from langgraph.graph import StateGraph, END
from langgraph.prebuilt.tool_node import ToolNode

from src.haive.core.engine.agent.agent import Agent, AgentConfig, register_agent
from src.haive.core.models.llm.base import AzureLLMConfig
from src.haive.core.engine.aug_llm import AugLLMConfig
from src.haive.core.graph.NodeFactory import NodeFactory
from src.haive.core.graph.tool_injector import (
    ToolInjector, 
    state_tool, 
    store_tool, 
    hybrid_tool
)

# Import modules for React agent
from src.haive.core.graph.routing import (
    Router,
    RoutingCondition,
    Route,
    has_tool_calls,
    is_last_step
)
from src.haive.core.graph.retry import (
    RetryPolicy,
    execute_with_retry
)
from src.haive.core.graph.tool_config import (
    ToolConfig,
    NodeConfig,
    process_tools
)


# =============================================
# State Schemas
# =============================================

class ReactAgentSchema(BaseModel):
    """Base schema for React Agent state."""
    messages: List[Any] = Field(default_factory=list)
    remaining_steps: int = Field(default=10)
    is_last_step: bool = Field(default=False)
    has_tool_calls: bool = Field(default=False)


class ReactAgentStructuredSchema(ReactAgentSchema):
    """Schema for React Agent with structured output."""
    structured_response: Optional[Any] = Field(default=None)


# =============================================
# React Agent Config
# =============================================
@register_agent(ReactAgentConfig)
class ReactAgentConfig(AgentConfig):
    """Configuration for a React Agent."""
    
    # Node names
    reasoning_node_name: str = Field(
        default="agent", 
        description="Name for the reasoning node"
    )
    
    tool_node_name: str = Field(
        default="tools", 
        description="Name for the tool execution node"
    )
    
    structured_output_node_name: str = Field(
        default="structured_output", 
        description="Name for the structured output node"
    )
    
    # Tools configuration
    tools: List[Union[BaseTool, StructuredTool, Callable, ToolConfig]] = Field(
        default_factory=list, 
        description="List of tools available to the agent"
    )
    
    tool_choice: str = Field(
        default="auto", 
        description="Tool choice strategy: 'auto', 'any', or 'required'"
    )
    
    # Tool error handling
    handle_tool_errors: Union[bool, str, Callable] = Field(
        default=True,
        description="How to handle tool errors"
    )
    
    # Custom nodes
    custom_nodes: List[NodeConfig] = Field(
        default_factory=list,
        description="Custom nodes to add to the graph"
    )
    
    # Custom routing
    custom_router: Optional[Router] = Field(
        default=None,
        description="Custom router configuration for complex routing"
    )
    
    # Structured output
    structured_output_model: Optional[Type[BaseModel]] = Field(
        default=None,
        description="Optional model for structured output"
    )
    
    structured_output_retry_policy: Optional[RetryPolicy] = Field(
        default=None,
        description="Retry policy for structured output generation"
    )
    
    # System prompt
    system_prompt: str = Field(
        default="""You are an intelligent agent that can use tools to help solve problems.
When you need information or want to perform an action, use the appropriate tool.
Think step-by-step and decide if you need to use any tools to fulfill the user's request.
If you need to use a tool, specify the tool name and parameters.
If you don't need a tool, simply respond directly to the user.""",
        description="System prompt for the agent"
    )
    
    # Iteration control
    max_iterations: int = Field(
        default=10,
        description="Maximum number of iterations"
    )
    
    # Override schema creation to detect structured output
    schema: Optional[Union[Type[BaseModel], Dict[str, Any]]] = Field(
        default=None,
        description="Optional explicit schema. If None, derived based on configuration."
    )
    
    @field_validator("schema", mode="before")
    def validate_schema(cls, v, info):
        """Auto-select the appropriate schema based on configuration."""
        if v is not None:
            return v
            
        # Use structured schema if structured output is configured
        if info.data.get("structured_output_model"):
            return ReactAgentStructuredSchema
        
        # Otherwise use the base schema
        return ReactAgentSchema

    @classmethod
    def from_scratch(cls, 
                   system_prompt: Optional[str] = None,
                   model: str = "gpt-4o", 
                   temperature: float = 0.7,
                   tools: Optional[List[Union[BaseTool, StructuredTool, Callable, ToolConfig]]] = None,
                   structured_output_model: Optional[Type[BaseModel]] = None,
                   custom_nodes: Optional[List[NodeConfig]] = None,
                   custom_router: Optional[Router] = None,
                   name: Optional[str] = None,
                   max_iterations: int = 10,
                   **kwargs) -> 'ReactAgentConfig':
        """
        Create a ReactAgentConfig from scratch with basic parameters.
        
        Args:
            system_prompt: Optional system prompt
            model: Model name to use
            temperature: Temperature for generation
            tools: Optional list of tools (can include ToolConfig for routing)
            structured_output_model: Optional model for structured output
            custom_nodes: Optional custom nodes to add to the graph
            custom_router: Optional custom router configuration
            name: Optional agent name
            max_iterations: Maximum number of iterations
            **kwargs: Additional configuration options
            
        Returns:
            ReactAgentConfig instance
        """
        # Use provided system prompt or default
        prompt_content = system_prompt or cls.__fields__["system_prompt"].default
        
        # Create prompt template
        messages = [
            SystemMessage(content=prompt_content),
            MessagesPlaceholder(variable_name="messages")
        ]
        prompt = ChatPromptTemplate.from_messages(messages)
        
        # Create LLM config
        llm_config = AzureLLMConfig(
            model=model,
            parameters={"temperature": temperature}
        )
        
        # Extract base tools from ToolConfig wrappers if needed
        processed_tools, _ = process_tools(tools or [])
        
        # Create AugLLM config
        aug_llm = AugLLMConfig(
            name=f"{name or 'react'}_llm",
            llm_config=llm_config,
            prompt_template=prompt,
            tools=processed_tools
        )
        
        # Create and return config
        return cls(
            name=name or "react_agent",
            engine=aug_llm,
            tools=tools or [],
            custom_nodes=custom_nodes or [],
            custom_router=custom_router,
            structured_output_model=structured_output_model,
            system_prompt=prompt_content,
            max_iterations=max_iterations,
            **kwargs
        )

    @classmethod
    def from_aug_llm(cls,
                     aug_llm: AugLLMConfig,
                     tools: Optional[List[Union[BaseTool, StructuredTool, Callable, ToolConfig]]] = None,
                     structured_output_model: Optional[Type[BaseModel]] = None,
                     custom_nodes: Optional[List[NodeConfig]] = None,
                     custom_router: Optional[Router] = None,
                     name: Optional[str] = None,
                     max_iterations: int = 10,
                     **kwargs) -> 'ReactAgentConfig':
        """
        Create a ReactAgentConfig using an existing AugLLMConfig.
        
        Args:
            aug_llm: Existing AugLLMConfig to use
            tools: Optional list of tools (can include ToolConfig for routing)
            structured_output_model: Optional model for structured output
            custom_nodes: Optional custom nodes to add to the graph
            custom_router: Optional custom router configuration
            name: Optional agent name
            max_iterations: Maximum number of iterations
            **kwargs: Additional configuration options
            
        Returns:
            ReactAgentConfig instance
        """
        # Update tools in AugLLMConfig if needed
        if tools:
            processed_tools, _ = process_tools(tools)
            
            # If the AugLLMConfig already has tools, we need to create a new one
            if aug_llm.tools:
                aug_llm = aug_llm.model_copy(update={"tools": processed_tools})
        
        # Create and return config
        return cls(
            name=name or "react_agent",
            engine=aug_llm,
            tools=tools or [],
            custom_nodes=custom_nodes or [],
            custom_router=custom_router,
            structured_output_model=structured_output_model,
            max_iterations=max_iterations,
            **kwargs
        )

    @classmethod
    def from_llm_config(cls,
                        llm_config: Union[AzureLLMConfig, Dict[str, Any]],
                        prompt_template: Optional[Union[str, ChatPromptTemplate]] = None,
                        tools: Optional[List[Union[BaseTool, StructuredTool, Callable, ToolConfig]]] = None,
                        structured_output_model: Optional[Type[BaseModel]] = None,
                        custom_nodes: Optional[List[NodeConfig]] = None,
                        custom_router: Optional[Router] = None,
                        name: Optional[str] = None,
                        max_iterations: int = 10,
                        **kwargs) -> 'ReactAgentConfig':
        """
        Create a ReactAgentConfig using an LLM configuration.
        
        Args:
            llm_config: LLM configuration (AzureLLMConfig or dict)
            prompt_template: Optional prompt template or system prompt string
            tools: Optional list of tools (can include ToolConfig for routing)
            structured_output_model: Optional model for structured output
            custom_nodes: Optional custom nodes to add to the graph
            custom_router: Optional custom router configuration
            name: Optional agent name
            max_iterations: Maximum number of iterations
            **kwargs: Additional configuration options
            
        Returns:
            ReactAgentConfig instance
        """
        # Handle LLM config
        if isinstance(llm_config, dict):
            llm_config = AzureLLMConfig(**llm_config)
        
        # Handle prompt template
        if prompt_template is None:
            # Use default system prompt
            system_prompt = cls.__fields__["system_prompt"].default
            prompt_template = ChatPromptTemplate.from_messages([
                SystemMessage(content=system_prompt),
                MessagesPlaceholder(variable_name="messages")
            ])
        elif isinstance(prompt_template, str):
            # Convert string to template
            prompt_template = ChatPromptTemplate.from_messages([
                SystemMessage(content=prompt_template),
                MessagesPlaceholder(variable_name="messages")
            ])
        
        # Process tools
        processed_tools, _ = process_tools(tools or [])
        
        # Create AugLLM config
        aug_llm = AugLLMConfig(
            name=f"{name or 'react'}_llm",
            llm_config=llm_config,
            prompt_template=prompt_template,
            tools=processed_tools
        )
        
        # Create and return config
        return cls(
            name=name or "react_agent",
            engine=aug_llm,
            tools=tools or [],
            custom_nodes=custom_nodes or [],
            custom_router=custom_router,
            structured_output_model=structured_output_model,
            max_iterations=max_iterations,
            **kwargs
        )

In [ ]:
# src/haive/agents/react_agent/agent.py (Part 2 - Implementation)

# =============================================
# React Agent Implementation
# =============================================
@register_agent(ReactAgentConfig)
class ReactAgent(Agent[ReactAgentConfig]):
    """
    React agent that implements the ReAct pattern for reasoning and tool use.
    
    This agent follows a workflow:
    1. Reasoning about the input using an LLM
    2. Deciding when to use tools vs. providing a direct answer
    3. Executing tools as needed with flexible routing
    4. Optionally generating structured output
    
    The agent supports:
    - Custom nodes and routing
    - Tool-specific routing
    - State and store injection into tools
    - Structured output generation
    """
    
    def __init__(self, config: ReactAgentConfig):
        """Initialize the ReactAgent with the provided configuration."""
        # Process tools and extract configurations
        self.processed_tools, self.tool_configs = process_tools(config.tools)
        
        # Create tool node if tools are provided
        if self.processed_tools:
            self.tool_node = ToolNode(
                self.processed_tools,
                name=config.tool_node_name,
                handle_tool_errors=config.handle_tool_errors
            )
        else:
            self.tool_node = None
            
        # Initialize base agent
        super().__init__(config)
    
    def setup_workflow(self) -> None:
        """Set up the ReAct workflow with reasoning, tools, and structured output."""
        # Extract configuration
        has_tools = bool(self.config.tools)
        structured_output_model = self.config.structured_output_model
        
        # Create and add the iteration tracking node
        self.graph.add_node("iteration_tracker", self._create_iteration_tracker())
        
        # Create and add the reasoning node using NodeFactory
        self.graph.add_node(
            self.config.reasoning_node_name,
            NodeFactory.create_node(
                config=self.engine,
                input_mapping={"messages": "messages"},
                output_mapping={"output": "messages"},
                state_model=self.state_schema,
                next_node=None  # Will be determined by conditional routing
            )
        )
        
        # Set the entry point
        self.graph.set_entry_point("iteration_tracker")
        
        # Add edge from iteration tracker to reasoning
        self.graph.add_edge("iteration_tracker", self.config.reasoning_node_name)
        
        # If tools are provided, set up tool execution
        if has_tools:
            # Add the tool node
            self.graph.add_node(self.config.tool_node_name, self.tool_node)
            
            # Determine the post-tool destination
            tool_destination = END if not structured_output_model else self.config.structured_output_node_name
            
            # If there's a custom router, use it
            if self.config.custom_router:
                # Use the get_route method from the Router
                def router_func(state):
                    return self.config.custom_router.get_route(state)
                
                # Get all possible destinations from the router
                destinations = {
                    route.destination for route in self.config.custom_router.routes
                }
                destinations.add(self.config.custom_router.default_destination)
                
                # Create the routing map
                route_map = {
                    dest: END if dest == "END" else dest
                    for dest in destinations
                }
                
                # Add conditional edges
                self.graph.add_conditional_edges(
                    self.config.reasoning_node_name,
                    router_func,
                    route_map
                )
            else:
                # Use default router: tools or end
                self.graph.add_conditional_edges(
                    self.config.reasoning_node_name,
                    self._default_router,
                    {
                        self.config.tool_node_name: self.config.tool_node_name,
                        "end": tool_destination
                    }
                )
            
            # Set up tool routing
            tool_routes = {
                name: config.route_to 
                for name, config in self.tool_configs.items() 
                if config.route_to
            }
            
            if tool_routes:
                # Create a map of destinations to ensure they're valid
                destinations = {
                    route: route if route != "END" else END
                    for route in set(tool_routes.values())
                }
                
                # If iteration_tracker isn't in destinations, add it as default
                if "iteration_tracker" not in destinations:
                    destinations["iteration_tracker"] = "iteration_tracker"
                
                # Add conditional edges from tools
                self.graph.add_conditional_edges(
                    self.config.tool_node_name,
                    self._create_tool_router(tool_routes),
                    destinations
                )
            else:
                # Default route from tools back to iteration tracker
                self.graph.add_edge(self.config.tool_node_name, "iteration_tracker")
        else:
            # Without tools, go straight to the end or structured output
            self.graph.add_edge(
                self.config.reasoning_node_name,
                END if not structured_output_model else self.config.structured_output_node_name
            )
        
        # If structured output is configured, add that node
        if structured_output_model:
            self.graph.add_node(
                self.config.structured_output_node_name,
                self._create_structured_output_node()
            )
            
            # Add edge from structured output to END
            self.graph.add_edge(self.config.structured_output_node_name, END)
        
        # Add any custom nodes
        for node_config in self.config.custom_nodes:
            # Add the node
            self.graph.add_node(node_config.node_name, node_config.node_function)
            
            # Set up routing
            if isinstance(node_config.routes_to, str):
                # Simple direct routing
                dest = node_config.routes_to
                self.graph.add_edge(node_config.node_name, END if dest == "END" else dest)
            elif isinstance(node_config.routes_to, dict):
                # Mapping-based routing
                def create_mapping_router(mapping):
                    def router(state):
                        # Simple key mapping
                        for key, dest in mapping.items():
                            if key in state:
                                return key
                        # Default to first key
                        return list(mapping.keys())[0]
                    return router
                
                # Convert string "END" to actual END
                routes = {
                    k: END if v == "END" else v 
                    for k, v in node_config.routes_to.items()
                }
                
                # Add conditional edges
                self.graph.add_conditional_edges(
                    node_config.node_name,
                    create_mapping_router(node_config.routes_to),
                    routes
                )
            elif hasattr(node_config.routes_to, "get_route"):
                # Router-based routing (any object with get_route method)
                def create_router_func(router):
                    def router_func(state):
                        return router.get_route(state)
                    return router_func
                
                # Get all possible destinations
                if hasattr(node_config.routes_to, "routes"):
                    destinations = {
                        route.destination for route in node_config.routes_to.routes
                    }
                    destinations.add(node_config.routes_to.default_destination)
                else:
                    # If we can't determine destinations, use a default set
                    destinations = {"iteration_tracker", "agent", "tools", "END"}
                
                # Create the routing map
                route_map = {
                    dest: END if dest == "END" else dest
                    for dest in destinations
                }
                
                # Add conditional edges
                self.graph.add_conditional_edges(
                    node_config.node_name,
                    create_router_func(node_config.routes_to),
                    route_map
                )
    
    def _create_iteration_tracker(self) -> Callable:
        """Create a node to track iterations."""
        max_iterations = self.config.max_iterations
        
        def iteration_tracker(state: Dict[str, Any]) -> Dict[str, Any]:
            """Track iterations and check for max iterations reached."""
            # Initialize state fields if not set
            if "remaining_steps" not in state:
                state["remaining_steps"] = max_iterations
                state["is_last_step"] = False
                state["has_tool_calls"] = False
                return state
            
            # Decrement remaining steps
            state["remaining_steps"] -= 1
            
            # Check if we've reached the limit
            if state["remaining_steps"] <= 0:
                state["is_last_step"] = True
                # Add a message about reaching max iterations if needed
                messages = state.get("messages", [])
                if messages:
                    last_message = messages[-1]
                    if not (isinstance(last_message, AIMessage) and "maximum number of iterations" in last_message.content):
                        state["messages"].append(
                            AIMessage(content=f"I've reached the maximum number of iterations ({max_iterations}). "
                                     "I'll provide my best response based on what I've learned so far.")
                        )
            
            return state
        
        return iteration_tracker
    
    def _default_router(self, state: Dict[str, Any]) -> str:
        """Default router for the reasoning node."""
        # If we're on the last step, go to the end
        if state.get("is_last_step", False) or state.get("remaining_steps", 0) <= 0:
            return "end"
        
        # Check if the last message has tool calls
        has_tool_calls = state.get("has_tool_calls", False)
        
        # If the state doesn't have the flag, check manually
        if not has_tool_calls:
            messages = state.get("messages", [])
            if messages and isinstance(messages[-1], AIMessage):
                has_tool_calls = hasattr(messages[-1], "tool_calls") and bool(messages[-1].tool_calls)
                state["has_tool_calls"] = has_tool_calls
        
        # If we have tool calls, route to the tool node
        if has_tool_calls:
            return self.config.tool_node_name
        
        # Otherwise, go to the end
        return "end"
    
    def _create_tool_router(self, tool_routes: Dict[str, str]) -> Callable:
        """Create a router for the tool node."""
        
        def tool_router(state: Dict[str, Any]) -> str:
            """Route based on the name of the last executed tool."""
            messages = state.get("messages", [])
            
            # Find the most recent tool message
            for msg in reversed(messages):
                if isinstance(msg, ToolMessage) and hasattr(msg, "name"):
                    # If we have a route for this tool, use it
                    if msg.name in tool_routes:
                        return tool_routes[msg.name]
                    break
            
            # Default to iteration tracker
            return "iteration_tracker"
        
        return tool_router
    
    def _create_structured_output_node(self) -> Callable:
        """Create a node for generating structured output."""
        model = self.config.engine.llm_config.instantiate_llm()
        structured_output_model = self.config.structured_output_model
        retry_policy = self.config.structured_output_retry_policy
        
        def structured_output_node(state: Dict[str, Any]) -> Dict[str, Any]:
            """Generate structured output from the conversation with retry support."""
            messages = state.get("messages", [])
            
            # Get model with structured output
            model_with_structured_output = model.with_structured_output(structured_output_model)
            
            # Define the generation function
            def generate_structured_output():
                """Generate structured output from messages."""
                return model_with_structured_output.invoke(messages)
            
            # Execute with retry if policy is provided
            if retry_policy:
                try:
                    response = execute_with_retry(
                        generate_structured_output,
                        retry_policy=retry_policy,
                        fallback_result={}
                    )
                    state["structured_response"] = response
                except Exception as e:
                    # If retries failed, log error and return empty object
                    error_msg = f"Unable to generate structured response after {retry_policy.max_retries} attempts: {str(e)}"
                    state["messages"].append(AIMessage(content=error_msg))
                    state["structured_response"] = {}
            else:
                # No retry policy, simple execution
                try:
                    response = generate_structured_output()
                    state["structured_response"] = response
                except Exception as e:
                    # If generation fails, log error and return empty object
                    error_msg = f"Unable to generate structured response: {str(e)}"
                    state["messages"].append(AIMessage(content=error_msg))
                    state["structured_response"] = {}
            
            return state
        
        return structured_output_node


# =============================================
# Helper functions
# =============================================

def create_react_agent(
    system_prompt: Optional[str] = None,
    model: str = "gpt-4o",
    temperature: float = 0.7,
    tools: Optional[List[Union[BaseTool, StructuredTool, Callable, ToolConfig]]] = None,
    structured_output_model: Optional[Type[BaseModel]] = None,
    custom_nodes: Optional[List[NodeConfig]] = None,
    custom_router: Optional[Router] = None,
    name: Optional[str] = None,
    max_iterations: int = 10,
    visualize: bool = False,
    llm_config: Optional[Union[AzureLLMConfig, Dict[str, Any]]] = None,
    engine: Optional[AugLLMConfig] = None,
    **kwargs
) -> ReactAgent:
    """
    Create a React agent with the specified configuration.
    
    Args:
        system_prompt: Optional system prompt
        model: Model name to use
        temperature: Temperature for generation
        tools: Optional list of tools (can include ToolConfig for routing)
        structured_output_model: Optional model for structured output
        custom_nodes: Optional custom nodes to add to the graph
        custom_router: Optional custom router configuration
        name: Optional agent name
        max_iterations: Maximum number of iterations
        visualize: Whether to generate visualization
        llm_config: Optional explicit LLM configuration (overrides model and temperature)
        engine: Optional explicit AugLLMConfig to use (highest priority)
        **kwargs: Additional configuration options
        
    Returns:
        ReactAgent instance
    """
    # Determine the best way to create the config
    if engine:
        # Create config from existing AugLLMConfig
        config = ReactAgentConfig.from_aug_llm(
            aug_llm=engine,
            tools=tools,
            structured_output_model=structured_output_model,
            custom_nodes=custom_nodes,
            custom_router=custom_router,
            name=name,
            max_iterations=max_iterations,
            **kwargs
        )
    if llm_config:
        # Create config from LLM config
        config = ReactAgentConfig.from_llm_config(
            llm_config=llm_config,
            prompt_template=system_prompt,
            tools=tools,
            structured_output_model=structured_output_model,
            custom_nodes=custom_nodes,
            custom_router=custom_router,
            name=name,
            max_iterations=max_iterations,
            **kwargs
        )
    else:
        # Create config from scratch
        config = ReactAgentConfig.from_scratch(
            system_prompt=system_prompt,
            model=model,
            temperature=temperature,
            tools=tools,
            structured_output_model=structured_output_model,
            custom_nodes=custom_nodes,
            custom_router=custom_router,
            name=name,
            max_iterations=max_iterations,
            **kwargs
        )
    
    # Set visualization flag
    config.visualize = visualize
    
    # Build and return agent
    return config.build_agent()


# Re-export components from submodules for convenience
from src.haive.core.graph.routing import (
    create_router_from_pairs,
    create_custom_condition,
    create_tool_condition,
    create_keyword_condition,
    create_state_condition,
    has_tool_calls,
    is_last_step,
    contains_keywords
)

from src.haive.core.graph.retry import (
    create_retry_policy,
    create_exponential_backoff_policy,
    create_linear_backoff_policy
)

from src.haive.core.graph.tool_config import (
    configure_tool,
    create_node_config
)

In [ ]:
# examples/simple_react_agent_example.py

from typing import Dict, Any
from langchain_core.tools import tool
from langchain_core.messages import AIMessage

#from src.haive.agents.react_agent.agent import create_react_agent


# Define simple tools
@tool
def calculate(expression: str) -> str:
    """Calculate the result of a mathematical expression."""
    try:
        result = eval(expression)
        return f"The result of {expression} is {result}"
    except Exception as e:
        return f"Error calculating {expression}: {str(e)}"


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b


def main():
    """Run a minimal example to demonstrate the React agent."""
    # Create the agent with minimal configuration
    agent = create_react_agent(
        system_prompt="You are a helpful math assistant.",
        model="gpt-4o",
        tools=[calculate, multiply],
        name="math_agent"
    )
    
    # Run the agent with a simple question
    
    from IPython.display import Image, display
    
    #try:
    display(Image(agent.app.get_graph().draw_mermaid_png()))
    result = agent.run("What is 25 + 10, and then what is that result multiplied by 3?")
    #agent.app
    
    # Print the messages
    print("\n===== CONVERSATION =====\n")
    for i, message in enumerate(result.get("messages", [])):
        if isinstance(message, tuple):
            print(f"{i+1}. {message[0].upper()}: {message[1]}")
        elif hasattr(message, "content"):
            msg_type = getattr(message, "type", "AI").upper()
            print(f"{i+1}. {msg_type}: {message.content}")
        else:
            print(f"{i+1}. MESSAGE: {message}")


if __name__ == "__main__":
    main()